## Truerize Package Initialization

In [1]:
import os
import sys
import importlib

INIT_CODE = r'''
import os
os.environ.setdefault("TQDM_DISABLE", "1")

import sys
import subprocess
import importlib
import contextlib
import io
import warnings
import platform
import json
import html
import argparse
import base64
import csv
import random
import datetime
from pathlib import Path
from typing import Literal

warnings.filterwarnings("ignore")

PY = sys.version_info
OS = platform.system()

DEPENDENCY_GROUPS = [
    (
        "core",
        [
            ("tqdm", "tqdm", False, None),
            ("numpy", "numpy", False, None),
            ("pandas", "pandas", False, None),
            ("scipy", "scipy", False, None),
            ("joblib", "joblib", False, None),
            ("matplotlib", "matplotlib", False, None),
            ("seaborn", "seaborn", False, None),
            ("plotly", "plotly", False, None),
        ],
    ),
    (
        "data",
        [
            ("polars", "polars", False, (3, 8)),
            ("pyarrow", "pyarrow", False, None),
            ("duckdb", "duckdb", False, None),
        ],
    ),
    (
        "ml",
        [
            ("sklearn", "scikit-learn", False, None),
            ("xgboost", "xgboost", False, None),
            ("lightgbm", "lightgbm", False, None),
        ],
    ),
    (
        "xai",
        [
            ("shap", "shap", False, None),
            ("lime", "lime", False, None),
            ("pyod", "pyod", False, None),
        ],
    ),
    (
        "validation",
        [
            ("great_expectations", "great_expectations", True, (3, 8)),
            ("cerberus", "cerberus", False, None),
            ("pydantic", "pydantic", False, None),
        ],
    ),
    (
        "ui_and_docs",
        [
            ("docx", "python-docx", False, None),
            ("html2image", "html2image", True, None),
            ("IPython", "ipython", False, None),
            ("ipywidgets", "ipywidgets", False, None),
            ("panel", "panel", False, (3, 8)),
        ],
    ),
]

IMPORT_FAILURES = {}
INSTALL_FAILURES = {}


class _UnavailableDependency:
    def __init__(self, module_name, error):
        self._module_name = module_name
        self._error = error

    def __getattr__(self, name):
        raise ImportError(
            f"Optional dependency '{self._module_name}' is unavailable in this environment. "
            f"Original error: {self._error}"
        ) from self._error

    def __call__(self, *args, **kwargs):
        raise ImportError(
            f"Optional dependency '{self._module_name}' is unavailable in this environment. "
            f"Original error: {self._error}"
        ) from self._error


def _bootstrap_pip():
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "--version"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=30,
            check=False,
        )
        return
    except Exception:
        pass
    try:
        import ensurepip

        ensurepip.bootstrap(upgrade=True)
        return
    except Exception:
        pass
    try:
        subprocess.run(
            [sys.executable, "-m", "ensurepip", "--upgrade"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            timeout=60,
            check=False,
        )
    except Exception:
        pass


def _try_import(module_name):
    try:
        importlib.invalidate_caches()
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            return importlib.import_module(module_name)
    except Exception as exc:
        IMPORT_FAILURES[module_name] = exc
        return None


def _install_package(pip_name):
    name_map = {
        "scikit-learn": "sklearn",
        "python-docx": "docx",
        "html2image": "html2image",
    }
    module_name = name_map.get(pip_name, pip_name.replace("-", "_"))
    commands = [
        [sys.executable, "-m", "pip", "install", pip_name, "--quiet", "--no-warn-script-location"],
        [sys.executable, "-m", "pip", "install", pip_name, "--user", "--quiet", "--no-warn-script-location"],
        [sys.executable, "-m", "pip", "install", pip_name, "--quiet", "--ignore-requires-python"],
        [sys.executable, "-m", "pip", "install", pip_name, "--force-reinstall", "--quiet", "--no-deps"],
    ]

    for command in commands:
        try:
            subprocess.run(
                command,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
                timeout=600,
                check=False,
            )
            if _try_import(module_name) is not None:
                return True
        except Exception:
            continue
    return False


def _install_dependency_groups():
    _bootstrap_pip()
    for _, dependencies in DEPENDENCY_GROUPS:
        for module_name, pip_name, optional, min_py in dependencies:
            if min_py and PY < min_py:
                continue

            imported = _try_import(module_name)
            if imported is not None:
                IMPORT_FAILURES.pop(module_name, None)
                continue

            installed = _install_package(pip_name)
            imported = _try_import(module_name)

            if imported is not None:
                IMPORT_FAILURES.pop(module_name, None)
                continue

            error = IMPORT_FAILURES.get(module_name)
            INSTALL_FAILURES[module_name] = error
            if not optional:
                raise ImportError(
                    f"[truerize] Required dependency '{pip_name}' could not be loaded."
                ) from error


with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    _install_dependency_groups()

import tqdm
import tqdm.std
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="IProgress not found.*")
    import tqdm.auto


class _SilentTqdm:
    @staticmethod
    def format_interval(seconds):
        return "00:00"

    @staticmethod
    def format_meter(*args, **kwargs):
        return ""

    @staticmethod
    def format_sizeof(num, suffix="", divisor=1000):
        try:
            return str(round(float(num), 2))
        except Exception:
            return "0"

    @staticmethod
    def status_printer(*args, **kwargs):
        return lambda *a, **kw: None

    def __init__(self, iterable=None, *args, **kwargs):
        self.iterable = iterable
        self.total = kwargs.get("total", 0)
        self.n = 0
        self.disable = True
        self.desc = kwargs.get("desc", "")

    def __iter__(self):
        return iter(self.iterable or [])

    def update(self, n=0):
        self.n += n or 0

    def close(self):
        return None

    def refresh(self, *args, **kwargs):
        return None

    def set_description(self, *args, **kwargs):
        return None

    def set_postfix(self, *args, **kwargs):
        return None

    def reset(self, *args, **kwargs):
        self.n = 0

    def display(self, *args, **kwargs):
        return None

    def clear(self, *args, **kwargs):
        return None

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        self.close()
        return False


def _silent_trange(*args, **kwargs):
    try:
        iterable = range(*args)
    except Exception:
        iterable = []
    return _SilentTqdm(iterable=iterable, **kwargs)


tqdm.tqdm = _SilentTqdm
tqdm.std.tqdm = _SilentTqdm
tqdm.trange = _silent_trange
tqdm.auto.tqdm = _SilentTqdm
tqdm.auto.trange = _silent_trange

try:
    import tqdm.autonotebook

    tqdm.autonotebook.tqdm = _SilentTqdm
    tqdm.autonotebook.trange = _silent_trange
except Exception:
    pass

try:
    import tqdm.notebook

    tqdm.notebook.tqdm = _SilentTqdm
    tqdm.notebook.tqdm_notebook = _SilentTqdm
    tqdm.notebook.trange = _silent_trange
except Exception:
    pass

for module_name in ("tqdm", "tqdm.std", "tqdm.auto", "tqdm.autonotebook", "tqdm.notebook"):
    module = sys.modules.get(module_name)
    if module is not None:
        try:
            module.tqdm = _SilentTqdm
        except Exception:
            pass
        try:
            module.trange = _silent_trange
        except Exception:
            pass

import numpy as np
import pandas as pd
import scipy
from scipy import stats
from scipy.stats import gaussian_kde

import polars as pl
import polars.selectors as cs
import pyarrow as pa
import duckdb

from sklearn import metrics, preprocessing
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_squared_error
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_class_weight

import xgboost as xgb
from xgboost import XGBClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier, LGBMRegressor

import shap
import lime
from lime import lime_tabular
from lime.lime_tabular import LimeTabularExplainer
import pyod
from pyod.models.iforest import IForest

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as plotly_express
import plotly.graph_objects as plotly_graph_objects
px = plotly_express
go = plotly_graph_objects
sp = scipy

import joblib
import panel
import IPython
import ipywidgets as widgets

try:
    from docx import Document
    from docx.shared import Inches, Pt
except Exception as exc:
    raise ImportError("[truerize] python-docx loaded but submodules could not be imported.") from exc

try:
    import html2image as _html2image_module

    Html2Image = getattr(_html2image_module, "Html2Image", None)
except Exception as exc:
    IMPORT_FAILURES["html2image"] = exc
    _html2image_module = _UnavailableDependency("html2image", exc)
    Html2Image = _UnavailableDependency("html2image.Html2Image", exc)

try:
    from cerberus import Validator
except Exception as exc:
    raise ImportError("[truerize] cerberus loaded but Validator could not be imported.") from exc

try:
    from pydantic import (
        BaseModel,
        ConfigDict,
        StrictFloat,
        StrictInt,
        StrictStr,
        ValidationError,
        create_model,
        field_validator,
    )
except Exception:
    from pydantic import BaseModel, StrictFloat, StrictInt, StrictStr, ValidationError, create_model, validator as field_validator

    class ConfigDict(dict):
        pass

from IPython import get_ipython as _get_ipython


def get_ipython():
    return _get_ipython()


try:
    import great_expectations as gx
    from great_expectations.core.batch import Batch
    from great_expectations.execution_engine import PandasExecutionEngine
    from great_expectations.render.renderer import ExpectationSuitePageRenderer, ValidationResultsPageRenderer
    from great_expectations.render.view import DefaultJinjaPageView
    from great_expectations.validator.validator import Validator as GXValidator

    try:
        import great_expectations.validator.validation_graph as _gx_validation_graph

        _gx_validation_graph.tqdm = _SilentTqdm
    except Exception:
        pass

    try:
        import great_expectations.validator.metrics_calculator as _gx_metrics_calculator

        _gx_metrics_calculator.tqdm = _SilentTqdm
    except Exception:
        pass

    try:
        import great_expectations.validator.validator as _gx_validator_module

        _gx_validator_module.tqdm = _SilentTqdm
    except Exception:
        pass
except Exception as exc:
    IMPORT_FAILURES["great_expectations"] = exc
    gx = _UnavailableDependency("great_expectations", exc)
    DefaultJinjaPageView = _UnavailableDependency("great_expectations.render.view", exc)
    ValidationResultsPageRenderer = _UnavailableDependency("great_expectations.render.renderer", exc)
    ExpectationSuitePageRenderer = _UnavailableDependency("great_expectations.render.renderer", exc)
    Batch = _UnavailableDependency("great_expectations.core.batch", exc)
    PandasExecutionEngine = _UnavailableDependency("great_expectations.execution_engine", exc)
    GXValidator = _UnavailableDependency("great_expectations.validator.validator", exc)


def get_cwd():
    return Path.cwd().resolve()


def make_project_folders(folders=("data", "images", "reports", "outputs", "models", "logs")):
    root = Path.cwd()
    for folder in folders:
        (root / folder).mkdir(parents=True, exist_ok=True)


def dependency_report():
    items = {
        "numpy": np,
        "pandas": pd,
        "scipy": sp,
        "matplotlib": matplotlib,
        "seaborn": sns,
        "plotly": plotly_express,
        "polars": pl,
        "pyarrow": pa,
        "duckdb": duckdb,
        "scikit-learn": sys.modules.get("sklearn"),
        "xgboost": xgb,
        "lightgbm": lgb,
        "shap": shap,
        "lime": lime,
        "pyod": pyod,
        "great_expectations": None if isinstance(gx, _UnavailableDependency) else gx,
        "cerberus": Validator,
        "pydantic": BaseModel,
        "python-docx": Document,
        "html2image": None if isinstance(Html2Image, _UnavailableDependency) else Html2Image,
        "IPython": IPython,
        "ipywidgets": widgets,
        "panel": panel,
        "joblib": joblib,
    }
    ok = sum(value is not None for value in items.values())
    fail = len(items) - ok
    print(f"\n  Truerize | Python {'.'.join(str(x) for x in PY[:3])} | {OS}")
    print(f"  {ok} loaded  |  {fail} missing\n")
    for name, module in items.items():
        status = "OK     " if module is not None else "MISSING"
        print(f"  {status}  {name}")
    print()


def reinstall_failed():
    for _, dependencies in DEPENDENCY_GROUPS:
        for module_name, pip_name, optional, min_py in dependencies:
            if min_py and PY < min_py:
                continue
            if _try_import(module_name) is None:
                installed = _install_package(pip_name)
                if not installed and not optional:
                    raise ImportError(f"[truerize] Unable to reinstall required dependency: {pip_name}")


__all__ = [
    "os",
    "sys",
    "subprocess",
    "importlib",
    "contextlib",
    "io",
    "warnings",
    "platform",
    "json",
    "html",
    "argparse",
    "base64",
    "csv",
    "Path",
    "Literal",
    "random",
    "datetime",
    "np",
    "pd",
    "sp",
    "pl",
    "cs",
    "pa",
    "duckdb",
    "stats",
    "gaussian_kde",
    "scipy",
    "metrics",
    "preprocessing",
    "RandomForestClassifier",
    "RandomForestRegressor",
    "LinearRegression",
    "LogisticRegression",
    "train_test_split",
    "cross_val_score",
    "GridSearchCV",
    "StratifiedKFold",
    "Pipeline",
    "StandardScaler",
    "MinMaxScaler",
    "LabelEncoder",
    "accuracy_score",
    "mean_squared_error",
    "classification_report",
    "confusion_matrix",
    "compute_class_weight",
    "SVC",
    "xgb",
    "XGBClassifier",
    "lgb",
    "LGBMRegressor",
    "LGBMClassifier",
    "shap",
    "lime",
    "lime_tabular",
    "LimeTabularExplainer",
    "pyod",
    "IForest",
    "matplotlib",
    "plt",
    "sns",
    "plotly_express",
    "plotly_graph_objects",
    "px",
    "go",
    "panel",
    "gx",
    "DefaultJinjaPageView",
    "ValidationResultsPageRenderer",
    "ExpectationSuitePageRenderer",
    "Batch",
    "PandasExecutionEngine",
    "GXValidator",
    "Validator",
    "Document",
    "Inches",
    "Pt",
    "Html2Image",
    "joblib",
    "BaseModel",
    "ConfigDict",
    "StrictFloat",
    "StrictInt",
    "StrictStr",
    "create_model",
    "field_validator",
    "ValidationError",
    "tqdm",
    "widgets",
    "IPython",
    "get_ipython",
    "get_cwd",
    "make_project_folders",
    "dependency_report",
    "reinstall_failed",
]

print("truerize package and __init__.py file created successfully and imported.")

if IMPORT_FAILURES:
    failed_modules = ", ".join(sorted(IMPORT_FAILURES))
    warnings.warn(
        "truerize imported with limited optional dependency support. "
        f"Unavailable or incompatible modules: {failed_modules}.",
        RuntimeWarning,
        stacklevel=2,
    )
'''

pkg = os.path.join(os.getcwd(), "truerize")
os.makedirs(pkg, exist_ok=True)
with open(os.path.join(pkg, "__init__.py"), "w", encoding="utf-8") as f:
    f.write(INIT_CODE.strip() + "\n")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

sys.modules.pop("truerize", None)
importlib.invalidate_caches()

import truerize

truerize package and __init__.py file created successfully and imported.


In [2]:
print("Current Working Directory:")
print(truerize.os.getcwd())


Current Working Directory:
/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction


In [3]:
import sys
print(sys.executable)


/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/.venv/bin/python


## Configure Project Paths

This cell defines the project directory, dataset location, random seed, and output folders used to store validation reports and model results.

In [4]:
# APP_EXPORT_START
Path = truerize.Path
truerize.os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path(truerize.os.getcwd()).resolve()
DATASET_PATH = BASE_DIR / "dataset" / "Airbnb_Open_Data.csv"
SEED = 42

OUTPUT_FOLDERS = [
    "GE",
    "cerberus_reports",
    "duckdb_reports",
    "pydantic_reports",
    "polars_reports",
    "root_cause_reports",
    "drift_reports",
    "ks_reports",
    "outputs/models",
    "outputs/lime",
    "outputs/shap",
    "outputs/xai_report",
    "outputs/runtime",
    "outputs/monitoring",
]
# APP_EXPORT_END

## Create Output Folders

This cell creates all required project output folders if they do not already exist and displays their paths.

In [5]:
# APP_EXPORT_START
def ensure_output_folders():
    for folder in OUTPUT_FOLDERS:
        (BASE_DIR / folder).mkdir(parents=True, exist_ok=True)
# APP_EXPORT_END

ensure_output_folders()
print("Created/Exists output folders:")
for folder in OUTPUT_FOLDERS:
    print(BASE_DIR / folder)

Created/Exists output folders:
/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/GE
/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/cerberus_reports
/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/duckdb_reports
/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/pydantic_reports
/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/polars_reports
/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/root_cause_reports
/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/drift_reports
/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/ks_reports
/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/models
/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/lime
/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outpu

## Enable Cell Output Logging

This cell logs the output of every notebook cell to a text file for tracking and debugging purposes.

In [6]:
def enable_cell_output_logging():
    ip = truerize.get_ipython()
    if ip is None:
        print("Cell output logging is only available inside an IPython notebook.")
        return

    folder = "cell_outputs"
    Path(folder).mkdir(exist_ok=True)
    log_file = open(Path(folder) / "cell_outputs.txt", "w", encoding="utf-8")
    cell_counter = [0]

    class Tee:
        def __init__(self, stream):
            self.stream = stream

        def write(self, text):
            self.stream.write(text)
            self.stream.flush()
            log_file.write(text)
            log_file.flush()

        def flush(self):
            self.stream.flush()
            log_file.flush()

        def __getattr__(self, name):
            return getattr(self.stream, name)

    old_displayhook = ip.displayhook.write_format_data

    def pre_run_cell(_):
        cell_counter[0] += 1
        log_file.write(f"\n{'=' * 40}\nCell {cell_counter[0]}\n{'=' * 40}\n")
        log_file.flush()

    def displayhook(format_dict, metadata=None):
        text_output = format_dict.get("text/plain")
        if text_output:
            log_file.write(text_output + "\n")
            log_file.flush()
        return old_displayhook(format_dict, metadata)

    truerize.sys.stdout = Tee(truerize.sys.stdout)
    truerize.sys.stderr = Tee(truerize.sys.stderr)
    ip.displayhook.write_format_data = displayhook
    ip.events.register("pre_run_cell", pre_run_cell)
    print(f"Logging to: {folder}/cell_outputs.txt")

enable_cell_output_logging()


Logging to: cell_outputs/cell_outputs.txt


## Load the Raw Dataset

This cell configures Polars display settings, loads the raw Airbnb dataset, and displays its schema, shape, sample records, and null value summary.

In [7]:
# APP_EXPORT_START
def _configure_polars():
    "Set Polars display configuration."
    truerize.pl.Config.set_tbl_cols(12)
    truerize.pl.Config.set_tbl_rows(6)
    truerize.pl.Config.set_tbl_width_chars(220)
    truerize.pl.Config.set_tbl_formatting("ASCII_FULL")


def load_raw_dataset():
    ensure_output_folders()
    return truerize.pl.scan_csv(
        DATASET_PATH,
        infer_schema_length=10000,
        ignore_errors=True,
    ).collect()
# APP_EXPORT_END

_configure_polars()
raw_df = load_raw_dataset()
print(raw_df.schema)
print('Shape:', raw_df.shape)
print(raw_df.head(5))
print('Null counts:')
print(raw_df.null_count())

Schema({'id': Int64, 'NAME': String, 'host id': Int64, 'host_identity_verified': String, 'host name': String, 'neighbourhood group': String, 'neighbourhood': String, 'lat': Float64, 'long': Float64, 'country': String, 'country code': String, 'instant_bookable': Boolean, 'cancellation_policy': String, 'room type': String, 'Construction year': Int64, 'price': String, 'service fee': String, 'minimum nights': Int64, 'number of reviews': Int64, 'last review': String, 'reviews per month': Float64, 'review rate number': Int64, 'calculated host listings count': Int64, 'availability 365': Int64, 'house_rules': String, 'license': String})
Shape: (102599, 26)
shape: (5, 26)
+---------+------------------------+-------------+-----------------------+-----------+---------------+-----+-------------------+--------------------+-----------------+------------------+-----------------------+---------+
| id      | NAME                   | host id     | host_identity_verifie | host name | neighbourhood | ... 

## Normalize the Dataset

This cell standardizes column names, cleans monetary values and dates, and normalizes neighbourhood group names for consistent data processing.

In [8]:
# APP_EXPORT_START
COLUMN_RENAME_MAP = {
    "id": "id",
    "NAME": "name",
    "host id": "host_id",
    "host_identity_verified": "host_identity_verified",
    "host name": "host_name",
    "neighbourhood group": "neighbourhood_group",
    "neighbourhood": "neighbourhood",
    "lat": "lat",
    "long": "long",
    "country": "country",
    "country code": "country_code",
    "instant_bookable": "instant_bookable",
    "cancellation_policy": "cancellation_policy",
    "room type": "room_type",
    "Construction year": "construction_year",
    "price": "price",
    "service fee": "service_fee",
    "minimum nights": "minimum_nights",
    "number of reviews": "number_of_reviews",
    "last review": "last_review",
    "reviews per month": "reviews_per_month",
    "review rate number": "review_rate_number",
    "calculated host listings count": "calculated_host_listings_count",
    "availability 365": "availability_365",
    "house_rules": "house_rules",
    "license": "license",
}

NEIGHBOURHOOD_GROUP_FIXES = {
    "manhatan": "Manhattan",
    "brookln": "Brooklyn",
}


def _clean_money_column(df, column):
    return df.with_columns(
        truerize.pl.col(column)
        .str.replace_all(r"[\$,]", "")
        .str.strip_chars()
        .cast(truerize.pl.Float64, strict=False)
        .alias(column)
    )


def normalize_raw_dataset(df):
    working = df.rename(COLUMN_RENAME_MAP)

    working = _clean_money_column(working, "price")
    working = _clean_money_column(working, "service_fee")

    working = working.with_columns(
        truerize.pl.col("last_review")
        .str.strptime(truerize.pl.Date, format="%m/%d/%Y", strict=False)
        .alias("last_review")
    )

    working = working.with_columns(
        truerize.pl.col("neighbourhood_group")
        .str.strip_chars()
        .str.to_lowercase()
        .replace(NEIGHBOURHOOD_GROUP_FIXES)
        .alias("neighbourhood_group_clean")
    )
    # capitalize anything not in the fix map that's still lowercase (e.g. "brooklyn" -> "Brooklyn")
    working = working.with_columns(
        truerize.pl.when(truerize.pl.col("neighbourhood_group_clean").is_in(list(NEIGHBOURHOOD_GROUP_FIXES.values())))
        .then(truerize.pl.col("neighbourhood_group_clean"))
        .otherwise(truerize.pl.col("neighbourhood_group_clean").str.to_titlecase())
        .alias("neighbourhood_group_clean")
    )

    return working
# APP_EXPORT_END

raw_df = normalize_raw_dataset(raw_df)
print(raw_df.schema)
print(raw_df.select(["price", "service_fee", "last_review", "neighbourhood_group", "neighbourhood_group_clean"]).head(10))
print("neighbourhood_group_clean value counts:")
print(raw_df.group_by("neighbourhood_group_clean").len().sort("len", descending=True))

Schema({'id': Int64, 'name': String, 'host_id': Int64, 'host_identity_verified': String, 'host_name': String, 'neighbourhood_group': String, 'neighbourhood': String, 'lat': Float64, 'long': Float64, 'country': String, 'country_code': String, 'instant_bookable': Boolean, 'cancellation_policy': String, 'room_type': String, 'construction_year': Int64, 'price': Float64, 'service_fee': Float64, 'minimum_nights': Int64, 'number_of_reviews': Int64, 'last_review': Date, 'reviews_per_month': Float64, 'review_rate_number': Int64, 'calculated_host_listings_count': Int64, 'availability_365': Int64, 'house_rules': String, 'license': String, 'neighbourhood_group_clean': String})
shape: (10, 5)
+--------+-------------+-------------+---------------------+---------------------------+
| price  | service_fee | last_review | neighbourhood_group | neighbourhood_group_clean |
| ---    | ---         | ---         | ---                 | ---                       |
| f64    | f64         | date        | str  

## Validate Data with Polars

This cell detects duplicate listings and validates the dataset against predefined business rules using Polars, then saves the validation results.

In [9]:
# APP_EXPORT_START
ALLOWED_ROOM_TYPES = {"Private room", "Hotel room", "Entire home/apt", "Shared room"}
ALLOWED_CANCELLATION_POLICIES = {"moderate", "strict", "flexible"}
ALLOWED_HOST_IDENTITY = {"verified", "unconfirmed"}
ALLOWED_NEIGHBOURHOOD_GROUPS = {"Brooklyn", "Manhattan", "Queens", "Bronx", "Staten Island"}


def _row_validation_errors(row, clean=False):
    errors = []

    group_col = "neighbourhood_group_clean" if clean else "neighbourhood_group"

    if row.get("id") is None:
        errors.append("missing id")

    if row.get("price") is None or (row.get("price") is not None and row["price"] < 0):
        errors.append("invalid price")

    if row.get("service_fee") is None or (row.get("service_fee") is not None and row["service_fee"] < 0):
        errors.append("invalid service_fee")

    if row.get("minimum_nights") is not None and row["minimum_nights"] < 0:
        errors.append("negative minimum_nights")

    if row.get("availability_365") is not None and not (0 <= row["availability_365"] <= 365):
        errors.append("availability_365 out of range")

    if row.get("lat") is not None and not (-90 <= row["lat"] <= 90):
        errors.append("lat out of range")

    if row.get("long") is not None and not (-180 <= row["long"] <= 180):
        errors.append("long out of range")

    room_type = row.get("room_type")
    if room_type is not None and room_type not in ALLOWED_ROOM_TYPES:
        errors.append("invalid room_type")

    cancellation_policy = row.get("cancellation_policy")
    if cancellation_policy is not None and cancellation_policy not in ALLOWED_CANCELLATION_POLICIES:
        errors.append("invalid cancellation_policy")

    host_identity = row.get("host_identity_verified")
    if host_identity is not None and host_identity not in ALLOWED_HOST_IDENTITY:
        errors.append("invalid host_identity_verified")

    group_value = row.get(group_col)
    if group_value is not None and group_value not in ALLOWED_NEIGHBOURHOOD_GROUPS:
        errors.append("invalid neighbourhood_group")

    return errors
# APP_EXPORT_START
def detect_duplicate_listings(df, name_prefix="raw"):
    dup_check_cols = [c for c in df.columns if c not in ("id",)]
    result = (
        df.with_columns(truerize.pl.col(dup_check_cols).is_duplicated().alias("is_duplicate_listing") if False else None)
    )
    # simpler and correct: flag rows that are duplicated across all columns except id
    mask = df.select(dup_check_cols).is_duplicated()
    flagged = df.with_columns(truerize.pl.Series("is_duplicate_listing", mask))
    output_path = BASE_DIR / "polars_reports" / f"{name_prefix}_duplicate_listings.csv"
    flagged.filter(truerize.pl.col("is_duplicate_listing")).write_csv(output_path)
    print(f"Duplicate listings ({name_prefix}): {mask.sum()} rows flagged, saved to {output_path}")
    return flagged
# APP_EXPORT_END

raw_dup_flagged = detect_duplicate_listings(raw_df, "raw")


def run_polars_validation(df, name_prefix="raw", clean=False):
    results = []
    seen_ids = set()
    for row in df.to_dicts():
        errors = _row_validation_errors(row, clean=clean)
        row_id = row.get("id")
        if row_id in seen_ids:
            errors.append("duplicate id")
        else:
            seen_ids.add(row_id)

        results.append(
            {
                "id": row_id,
                "status": "PASS" if not errors else "FAIL",
                "errors": "; ".join(errors) if errors else None,
            }
        )

    result_df = truerize.pl.DataFrame(results)
    output_path = BASE_DIR / "polars_reports" / f"{name_prefix}_polars_validation.csv"
    result_df.write_csv(output_path)
    return result_df
# APP_EXPORT_END

raw_polars = run_polars_validation(raw_df, "raw", clean=False)
print(raw_polars.group_by("status").len().sort("status"))
print("Sample failures:")
print(raw_polars.filter(truerize.pl.col("status") == "FAIL").head(10))

Duplicate listings (raw): 1082 rows flagged, saved to /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/polars_reports/raw_duplicate_listings.csv
shape: (2, 2)
+--------+-------+
| status | len   |
| ---    | ---   |
| str    | u32   |
+================+
| FAIL   | 4204  |
|--------+-------|
| PASS   | 98395 |
+--------+-------+
Sample failures:
shape: (10, 3)
+---------+--------+-------------------------------+
| id      | status | errors                        |
| ---     | ---    | ---                           |
| i64     | str    | str                           |
+==================================================+
| 1004098 | FAIL   | availability_365 out of range |
|---------+--------+-------------------------------|
| 1006307 | FAIL   | availability_365 out of range |
|---------+--------+-------------------------------|
| 1008516 | FAIL   | invalid neighbourhood_group   |
|---------+--------+-------------------------------|
| ...     | ...    | ...     

## Great Expectations Validation

This cell defines and executes the Great Expectations validation suite, generating HTML and JSON validation reports along with the expectation suite.

In [10]:
# APP_EXPORT_START
SOURCE_COLUMNS = [
    "id", "name", "host_id", "host_identity_verified", "host_name",
    "neighbourhood_group", "neighbourhood", "lat", "long", "country",
    "country_code", "instant_bookable", "cancellation_policy", "room_type",
    "construction_year", "price", "service_fee", "minimum_nights",
    "number_of_reviews", "last_review", "reviews_per_month",
    "review_rate_number", "calculated_host_listings_count",
    "availability_365", "house_rules", "license",
]


def _create_ge_paths(base_dir, name_prefix):
    json_path = truerize.os.path.join(base_dir, f"{name_prefix}_GE.json")
    html_path = truerize.os.path.join(base_dir, f"{name_prefix}_GE_report.html")
    suite_path = truerize.os.path.join(base_dir, f"{name_prefix}_GE_suite.html")
    return json_path, html_path, suite_path


def _escape(value) -> str:
    return truerize.html.escape("" if value is None else str(value))


def _prepare_ge_dataframe(df_pl, clean=False):
    df_pd = df_pl.to_pandas().copy()
    df_pd["last_review"] = df_pd["last_review"].astype(object)
    df_pd["review_consistency_ok"] = ~(
        (df_pd["number_of_reviews"] > 0) & (df_pd["last_review"].isna())
    )
    return df_pd


CLEAN_SOURCE_COLUMNS = [c for c in SOURCE_COLUMNS if c != "license"]

def _add_expectations(v, clean=False):
    group_col = "neighbourhood_group_clean" if clean else "neighbourhood_group"
    expected_columns = CLEAN_SOURCE_COLUMNS if clean else SOURCE_COLUMNS
    v.expect_table_columns_to_match_set(expected_columns + (["neighbourhood_group_clean"] if clean else []), exact_match=False)
    v.expect_table_row_count_to_be_between(min_value=1)

    v.expect_column_values_to_not_be_null("review_consistency_ok")
    v.expect_column_values_to_be_in_set("review_consistency_ok", [True], mostly=0.99)

    v.expect_column_values_to_not_be_null("id")
    v.expect_column_values_to_be_of_type("id", "int64")
    v.expect_column_values_to_be_unique("id")

    v.expect_column_values_to_not_be_null("host_id")
    v.expect_column_values_to_be_of_type("host_id", "int64")

    v.expect_column_values_to_not_be_null("price")
    v.expect_column_values_to_be_of_type("price", "float64")
    v.expect_column_values_to_be_between("price", 0, None)

    v.expect_column_values_to_not_be_null("service_fee")
    v.expect_column_values_to_be_of_type("service_fee", "float64")
    v.expect_column_values_to_be_between("service_fee", 0, None)

    # These columns are genuinely nullable in this dataset, so pandas/polars
    # represents them as float64 (NaN-safe) rather than int64. The expected
    # type is set to float64 here to reflect real data, not to mask an error —
    # the value-range checks below still enforce correctness.
    nullable_numeric_ranges = [
        ("minimum_nights", "float64", 0, 365),
        ("availability_365", "float64", 0, 365),
        ("review_rate_number", "float64", 1, 5),
        ("number_of_reviews", "float64", 0, None),
        ("calculated_host_listings_count", "float64", 0, None),
        ("construction_year", "float64", 2003, 2023),
    ]
    for column, expected_type, min_value, max_value in nullable_numeric_ranges:
        v.expect_column_values_to_be_of_type(column, expected_type)
        v.expect_column_values_to_be_between(column, min_value, max_value, mostly=0.99)

    # Truly non-nullable numeric columns stay strict int64/float64 as before
    v.expect_column_values_to_be_of_type("lat", "float64")
    v.expect_column_values_to_be_between("lat", -90, 90)
    v.expect_column_values_to_be_of_type("long", "float64")
    v.expect_column_values_to_be_between("long", -180, 180)
    v.expect_column_values_to_be_of_type("reviews_per_month", "float64")
    v.expect_column_values_to_be_between("reviews_per_month", 0, None)

    v.expect_column_values_to_be_of_type("last_review", "object")

    v.expect_column_values_to_be_in_set("room_type", list(ALLOWED_ROOM_TYPES))
    v.expect_column_values_to_be_in_set("cancellation_policy", list(ALLOWED_CANCELLATION_POLICIES), mostly=0.95)
    v.expect_column_values_to_be_in_set("host_identity_verified", list(ALLOWED_HOST_IDENTITY), mostly=0.95)
    v.expect_column_values_to_be_in_set(group_col, list(ALLOWED_NEIGHBOURHOOD_GROUPS), mostly=0.95)

    v.expect_column_values_to_be_of_type("instant_bookable", "bool")
    v.expect_column_values_to_be_in_set("instant_bookable", [True, False])
    v.expect_column_values_to_be_in_set("country", ["United States"], mostly=0.99)
    v.expect_column_values_to_be_in_set("country_code", ["US"], mostly=0.99)


def _save_json(result, path):
    with open(path, "w", encoding="utf-8") as file_obj:
        payload = result.to_json_dict() if hasattr(result, "to_json_dict") else result
        truerize.json.dump(payload, file_obj, indent=2, default=str)


def _save_html(content, path):
    with open(path, "w", encoding="utf-8") as file_obj:
        file_obj.write(content)


def _ge_rows_from_payload(payload):
    rows = []
    for item in payload.get("results", []):
        result = item.get("result", {})
        observed = result.get("observed_value")
        if observed is None and "unexpected_count" in result:
            observed = result.get("unexpected_count")
        details = result.get("details") or result.get("partial_unexpected_list") or ""
        if "unexpected_count" in result:
            details = f"Unexpected count: {result['unexpected_count']}"
        rows.append({
            "status": "Success" if item.get("success") else "Failed",
            "expectation": item.get("expectation_config", {}).get("type", "expectation"),
            "column": item.get("expectation_config", {}).get("kwargs", {}).get("column", "table"),
            "observed": observed,
            "details": details,
        })
    return rows


def _write_ge_standard_html(path, title, statistics, rows, warning=None, suite_mode=False):
    success_percent = statistics.get("success_percent", 0.0)
    evaluated = statistics.get("evaluated_expectations", len(rows))
    successful = statistics.get("successful_expectations", 0)
    unsuccessful = statistics.get("unsuccessful_expectations", 0)
    status_label = "Success" if unsuccessful == 0 else "Failed"
    badge_class = "success" if unsuccessful == 0 else "danger"
    page_title = "Expectation Validation Result" if not suite_mode else "Expectation Suite"
    nav_label = "Validations" if not suite_mode else "Suites"
    warning_block = ""
    if warning:
        warning_block = f"<div class='alert alert-warning mt-3' role='alert'>Fallback mode: {_escape(warning)}</div>"

    toc_items = []
    seen = []
    for row in rows:
        column = str(row.get("column", "table"))
        if column not in seen:
            seen.append(column)
            toc_items.append(column)

    def _slug(text):
        text = str(text).strip().lower().replace(" ", "-")
        return ''.join(ch for ch in text if ch.isalnum() or ch == '-') or 'section'

    left_nav = "".join(f"<li><a href='#{_slug(item)}'>{_escape(item)}</a></li>" for item in toc_items)

    grouped = {}
    for row in rows:
        grouped.setdefault(str(row.get("column", "table")), []).append(row)

    sections = []
    for column, items in grouped.items():
        body = []
        for row in items:
            icon = "&#10004;" if row.get("status") == "Success" else "&#10008;"
            icon_class = "text-success" if row.get("status") == "Success" else "text-danger"
            body.append(
                "<tr>"
                f"<td class='{icon_class}'>{icon}</td>"
                f"<td>{_escape(row.get('expectation'))}</td>"
                f"<td>{_escape(row.get('observed'))}</td>"
                f"<td>{_escape(row.get('details'))}</td>"
                "</tr>"
            )
        sections.append(
            f"<section id='{_slug(column)}' class='mb-4'>"
            f"<h4 class='section-title'>{_escape(column)}</h4>"
            "<div class='table-responsive'><table class='table table-dark table-striped table-bordered'>"
            "<thead><tr><th>Status</th><th>Expectation</th><th>Observed Value</th><th>Details</th></tr></thead>"
            f"<tbody>{''.join(body)}</tbody></table></div></section>"
        )

    html = f'''<!DOCTYPE html>
<html>
<head>
  <meta charset='utf-8'>
  <meta name='viewport' content='width=device-width, initial-scale=1.0'>
  <title>Data Docs - Great Expectations</title>
  <link rel='stylesheet' href='https://maxcdn.bootstrapcdn.com/bootstrap/4.3.1/css/bootstrap.min.css'>
  <style>
    body {{ background:#1b1f24; color:#f5f6f7; }}
    .gx-sidebar {{ position:fixed; top:0; left:0; width:300px; height:100vh; background:#161a1f; padding:20px; overflow:auto; border-right:1px solid #2d333b; }}
    .gx-main {{ margin-left:320px; padding:24px 28px; }}
    .gx-logo {{ font-size:28px; font-weight:700; color:#ff7a18; line-height:1.1; }}
    .gx-sub {{ color:#c9d1d9; font-size:13px; margin-bottom:18px; }}
    .summary-box {{ background:#e9ecef; color:#1b1f24; border-radius:4px; padding:18px; margin-bottom:24px; }}
    .stats-table td, .stats-table th {{ color:#f5f6f7; border-color:#3a3f44; }}
    .section-title {{ background:#e9ecef; color:#1b1f24; padding:10px 14px; border-radius:4px; margin:20px 0 10px; }}
    a {{ color:#8ab4ff; }}
    .toc-title {{ color:#f5f6f7; font-weight:600; margin-top:20px; }}
    .toc-list li {{ margin:8px 0; }}
  </style>
</head>
<body>
  <aside class='gx-sidebar'>
    <div class='gx-logo'>great<br>expectations</div>
    <div class='gx-sub'>Home / {nav_label} / {_escape(title)}</div>
    <h3>{page_title}</h3>
    <p>Evaluates whether a batch of data matches expectations.</p>
    <div class='toc-title'>Table of Contents</div>
    <ul class='toc-list'>{left_nav}</ul>
  </aside>
  <main class='gx-main'>
    <div class='summary-box'>
      <h3>Overview</h3>
      <div>Expectation Suite: <strong>{_escape(title)}</strong></div>
      <div>Data asset: None</div>
      <div>Status: <span class='text-{badge_class}'>{status_label}</span></div>
    </div>
    {warning_block}
    <h4>Statistics</h4>
    <table class='table table-dark table-bordered stats-table w-50'>
      <tbody>
        <tr><td>Evaluated Expectations</td><td>{evaluated}</td></tr>
        <tr><td>Successful Expectations</td><td>{successful}</td></tr>
        <tr><td>Unsuccessful Expectations</td><td>{unsuccessful}</td></tr>
        <tr><td>Success Percent</td><td>{success_percent:.2f}%</td></tr>
      </tbody>
    </table>
    {''.join(sections)}
  </main>
</body>
</html>'''
    _save_html(html, path)


def _render_suite_html(validator):
    return truerize.DefaultJinjaPageView().render(
        truerize.ExpectationSuitePageRenderer().render(validator.expectation_suite)
    )


def _render_validation_html(result):
    return truerize.DefaultJinjaPageView().render(
        truerize.ValidationResultsPageRenderer().render(result)
    )


def run_ge_validation(df_pl, name_prefix="raw", clean=False):
    truerize.warnings.filterwarnings("ignore")
    truerize.os.environ["TQDM_DISABLE"] = "1"

    ge_dir = truerize.os.path.join(str(BASE_DIR), "GE")
    truerize.os.makedirs(ge_dir, exist_ok=True)
    json_path, html_path, suite_path = _create_ge_paths(ge_dir, name_prefix)

    df_pd = _prepare_ge_dataframe(df_pl, clean=clean)

    suite = truerize.gx.core.ExpectationSuite(name=f"airbnb_suite_{name_prefix}")
    truerize.gx.get_context(mode="ephemeral")
    batch = truerize.Batch(data=df_pd)
    validator = truerize.GXValidator(
        execution_engine=truerize.PandasExecutionEngine(),
        batches=[batch],
        expectation_suite=suite,
    )

    _add_expectations(validator, clean=clean)
    result = validator.validate(result_format="COMPLETE")

    _save_json(result, json_path)
    _save_html(_render_suite_html(validator), suite_path)
    _save_html(_render_validation_html(result), html_path)

    result_dict = result.to_json_dict() if hasattr(result, "to_json_dict") else result
    stats = result_dict.get("statistics", {})
    evaluated = stats.get("evaluated_expectations", 0)
    successful = stats.get("successful_expectations", 0)
    success_percent = (successful / evaluated) * 100 if evaluated else 0

    print(f"GE Validation ({name_prefix}) Completed")
    print("HTML Report :", html_path)
    print("JSON Report :", json_path)
    print("Success %   :", round(success_percent, 2))
    print("Total Expectations:", evaluated)
    print("GE Suite Generated")
    print("HTML:", suite_path)

    return result_dict
# APP_EXPORT_END

raw_ge = run_ge_validation(raw_df, 'raw', clean=False)
print(raw_ge['statistics'])

GE Validation (raw) Completed
HTML Report : /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/GE/raw_GE_report.html
JSON Report : /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/GE/raw_GE.json
Success %   : 90.48
Total Expectations: 42
GE Suite Generated
HTML: /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/GE/raw_GE_suite.html
{'evaluated_expectations': 42, 'successful_expectations': 38, 'unsuccessful_expectations': 4, 'success_percent': 90.47619047619048}


## Report Generation Utilities

This cell defines helper functions to generate JSON and HTML validation reports and manages the output file paths for different validation frameworks.

In [11]:
# APP_EXPORT_START
def _write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(truerize.json.dumps(payload, indent=2, default=str), encoding="utf-8")


def _escape(value) -> str:
    return truerize.html.escape("" if value is None else str(value))


def _write_html_report(path, title, summary_text, columns, rows):
    body_rows = []
    for row in rows:
        body_rows.append("<tr>" + "".join(f"<td>{_escape(row.get(column))}</td>" for column in columns) + "</tr>")
    html = f'''<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>{_escape(title)}</title>
<style>
body {{ font-family: Arial, sans-serif; background:#f4f6fb; padding:20px; }}
.container {{ max-width: 1500px; margin:auto; }}
.header {{ background:#1f2d3d; color:white; padding:20px; border-radius:12px; margin-bottom:20px; }}
table {{ width:100%; border-collapse:collapse; background:white; border-radius:10px; overflow:hidden; }}
th, td {{ padding:10px; border-bottom:1px solid #eee; text-align:left; vertical-align:top; }}
thead {{ background:#263849; color:#fff; }}
</style>
</head>
<body>
<div class="container">
<div class="header">
<h2>{_escape(title)}</h2>
<p>{_escape(summary_text)}</p>
</div>
<table>
<thead><tr>{"".join(f"<th>{_escape(column)}</th>" for column in columns)}</tr></thead>
<tbody>{"".join(body_rows)}</tbody>
</table>
</div>
</body>
</html>'''
    path.write_text(html, encoding="utf-8")


def _validation_report_paths(name_prefix, validator_name):
    base_map = {
        "polars": BASE_DIR / "polars_reports",
        "duckdb": BASE_DIR / "duckdb_reports",
        "cerberus": BASE_DIR / "cerberus_reports",
        "pydantic": BASE_DIR / "pydantic_reports",
    }
    base = base_map[validator_name]
    return base / f"{name_prefix}_{validator_name}.json", base / f"{name_prefix}_{validator_name}_report.html"
# APP_EXPORT_END

## Validation Rules

This cell defines the validation rules to identify invalid records and generates domain-specific warnings based on business constraints.

In [12]:
# APP_EXPORT_START
def _contract_errors(row, clean=False):
    errors = []
    group_col = "neighbourhood_group_clean" if clean else "neighbourhood_group"

    if row.get("id") is None:
        errors.append("Missing id")

    price = row.get("price")
    if price is None or price < 0:
        errors.append("Invalid price")

    service_fee = row.get("service_fee")
    if service_fee is None or service_fee < 0:
        errors.append("Invalid service_fee")

    minimum_nights = row.get("minimum_nights")
    if minimum_nights is not None and (minimum_nights < 0 or minimum_nights > 365):
        errors.append("minimum_nights out of realistic range")

    availability = row.get("availability_365")
    if availability is not None and not (0 <= availability <= 365):
        errors.append("availability_365 out of range")

    lat = row.get("lat")
    if lat is not None and not (-90 <= lat <= 90):
        errors.append("lat out of range")

    long_ = row.get("long")
    if long_ is not None and not (-180 <= long_ <= 180):
        errors.append("long out of range")

    room_type = row.get("room_type")
    if room_type is not None and room_type not in ALLOWED_ROOM_TYPES:
        errors.append("Invalid room_type")

    cancellation_policy = row.get("cancellation_policy")
    if cancellation_policy is not None and cancellation_policy not in ALLOWED_CANCELLATION_POLICIES:
        errors.append("Invalid cancellation_policy")

    host_identity = row.get("host_identity_verified")
    if host_identity is not None and host_identity not in ALLOWED_HOST_IDENTITY:
        errors.append("Invalid host_identity_verified")

    group_value = row.get(group_col)
    if group_value is not None and group_value not in ALLOWED_NEIGHBOURHOOD_GROUPS:
        errors.append("Invalid neighbourhood_group")

    country = row.get("country")
    if country is not None and country != "United States":
        errors.append("country is not United States")

    number_of_reviews = row.get("number_of_reviews") or 0
    if number_of_reviews > 0 and row.get("last_review") is None:
        errors.append("Has reviews but last_review is missing")

    return errors


def _domain_warnings(row):
    warnings = []
    price = row.get("price") or 0
    if price >= 1000:
        warnings.append("premium price tier")
    if (row.get("number_of_reviews") or 0) == 0:
        warnings.append("no reviews yet")
    if (row.get("calculated_host_listings_count") or 0) >= 50:
        warnings.append("high-volume host")
    if (row.get("minimum_nights") or 0) >= 30:
        warnings.append("long minimum stay")
    if row.get("license") is None:
        warnings.append("no license on file")
    return warnings
# APP_EXPORT_END

## Polars Data Validation

This cell validates the dataset using Polars, applies predefined business rules, and generates JSON and HTML validation reports.

In [13]:
# APP_EXPORT_START
def run_polars_validation(df, name_prefix="raw", clean=False):
    records = []
    seen_ids = set()
    group_col = "neighbourhood_group_clean" if clean else "neighbourhood_group"
    for row in df.to_dicts():
        errors = _contract_errors(row, clean=clean)
        row_id = row.get("id")
        if row_id in seen_ids:
            errors.append("Duplicate id")
        else:
            seen_ids.add(row_id)
        records.append({
            "id": row_id,
            "price": row.get("price"),
            "room_type": row.get("room_type"),
            "neighbourhood_group": row.get(group_col),
            "status": "PASS" if not errors else "FAIL",
            "reason": "Valid" if not errors else " | ".join(errors),
            "warnings": _domain_warnings(row),
        })
    passed = sum(1 for r in records if r["status"] == "PASS")
    failed = len(records) - passed
    json_path, html_path = _validation_report_paths(name_prefix, "polars")
    _write_json(json_path, records)
    _write_html_report(
        html_path,
        f"Polars Validation ({name_prefix})",
        f"Total: {len(records)} | Passed: {passed} | Failed: {failed} | Success %: {round((passed / len(records)) * 100, 2) if records else 0}",
        ["id", "price", "room_type", "neighbourhood_group", "status", "reason", "warnings"],
        records,
    )
    return truerize.pl.DataFrame(records)
# APP_EXPORT_END

raw_polars = run_polars_validation(raw_df, "raw", clean=False)
print(raw_polars.group_by("status").len().sort("status"))

shape: (2, 2)
+--------+-------+
| status | len   |
| ---    | ---   |
| str    | u32   |
+================+
| FAIL   | 4266  |
|--------+-------|
| PASS   | 98333 |
+--------+-------+


## Cerberus Validation

This cell validates the dataset using a Cerberus schema and generates JSON, HTML validation reports, and an HTML schema report.

In [14]:
# APP_EXPORT_START
def _build_cerberus_schema(clean=False):
    group_col = "neighbourhood_group_clean" if clean else "neighbourhood_group"
    return {
        "id": {"type": "integer", "required": True, "nullable": False},
        "price": {"type": "float", "min": 0, "nullable": False},
        "service_fee": {"type": "float", "min": 0, "nullable": False},
        "minimum_nights": {"type": "integer", "min": 0, "max": 365, "nullable": True},
        "availability_365": {"type": "integer", "min": 0, "max": 365, "nullable": True},
        "lat": {"type": "float", "min": -90, "max": 90, "nullable": True},
        "long": {"type": "float", "min": -180, "max": 180, "nullable": True},
        "room_type": {"type": "string", "allowed": list(ALLOWED_ROOM_TYPES), "nullable": True},
        "cancellation_policy": {"type": "string", "allowed": list(ALLOWED_CANCELLATION_POLICIES), "nullable": True},
        group_col: {"type": "string", "allowed": list(ALLOWED_NEIGHBOURHOOD_GROUPS), "nullable": True},
    }


def run_cerberus_validation(df, name_prefix="raw", clean=False):
    schema = _build_cerberus_schema(clean=clean)
    validator = truerize.Validator(schema, allow_unknown=True)
    group_col = "neighbourhood_group_clean" if clean else "neighbourhood_group"

    results = []
    for row in df.to_dicts():
        coerced_row = dict(row)
        for field, rule in schema.items():
            if rule.get("type") == "float" and isinstance(coerced_row.get(field), int):
                coerced_row[field] = float(coerced_row[field])

        schema_valid = validator.validate(coerced_row)
        contract_errs = _contract_errors(row, clean=clean)
        cerberus_errs = [] if schema_valid else [f"{field}: {msg}" for field, msgs in validator.errors.items() for msg in msgs]
        all_errors = contract_errs + cerberus_errs

        results.append({
            "id": row.get("id"),
            "price": row.get("price"),
            "room_type": row.get("room_type"),
            "neighbourhood_group": row.get(group_col),
            "status": "PASS" if not all_errors else "FAIL",
            "reason": "Valid" if not all_errors else " | ".join(all_errors),
            "warnings": _domain_warnings(row),
        })

    passed = sum(1 for r in results if r["status"] == "PASS")
    failed = len(results) - passed
    json_path, html_path = _validation_report_paths(name_prefix, "cerberus")
    _write_json(json_path, results)
    _write_html_report(
        html_path,
        f"Cerberus Validation ({name_prefix})",
        f"Total: {len(results)} | Passed: {passed} | Failed: {failed} | Success %: {round((passed / len(results)) * 100, 2) if results else 0}",
        ["id", "price", "room_type", "neighbourhood_group", "status", "reason", "warnings"],
        results,
    )
    return results


def generate_cerberus_suite(df, name_prefix="raw"):
    rows = [{"Column": column, "Type": str(dtype)} for column, dtype in zip(df.columns, df.dtypes)]
    path = BASE_DIR / "cerberus_reports" / f"{name_prefix}_cerberus_suite.html"
    _write_html_report(path, "Cerberus Schema", f"Columns: {len(rows)}", ["Column", "Type"], rows)
    return path
# APP_EXPORT_END

raw_cerberus = run_cerberus_validation(raw_df, "raw", clean=False)
generate_cerberus_suite(raw_df, "raw")
print("Cerberus rows:", len(raw_cerberus))

Cerberus rows: 102599


In [15]:
# Run validation
raw_cerberus = run_cerberus_validation(raw_df, "raw", clean=False)

# Generate suite
suite_path = generate_cerberus_suite(raw_df, "raw")

# Get validation report paths
json_report_path, html_report_path = _validation_report_paths("raw", "cerberus")

# Print paths
print("Cerberus rows:", len(raw_cerberus))
print("Suite HTML:", suite_path)
print("Validation JSON Report:", json_report_path)
print("Validation HTML Report:", html_report_path)

Cerberus rows: 102599
Suite HTML: /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/cerberus_reports/raw_cerberus_suite.html
Validation JSON Report: /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/cerberus_reports/raw_cerberus.json
Validation HTML Report: /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/cerberus_reports/raw_cerberus_report.html


## Pydantic Validation

This cell validates the dataset using a Pydantic schema and generates JSON, HTML validation reports, and an HTML schema report.

In [16]:
# APP_EXPORT_START
class AirbnbListingSchema(truerize.BaseModel):
    model_config = truerize.ConfigDict(extra="allow")

    id: truerize.StrictInt
    host_id: truerize.StrictInt
    price: truerize.StrictFloat
    service_fee: truerize.StrictFloat
    minimum_nights: truerize.StrictInt | None = None
    availability_365: truerize.StrictInt | None = None
    lat: truerize.StrictFloat | None = None
    long: truerize.StrictFloat | None = None
    review_rate_number: truerize.StrictInt | None = None
    number_of_reviews: truerize.StrictInt | None = None
    reviews_per_month: truerize.StrictFloat | None = None
    calculated_host_listings_count: truerize.StrictInt | None = None
    construction_year: truerize.StrictInt | None = None
    instant_bookable: bool | None = None

    @truerize.field_validator("price", "service_fee")
    @classmethod
    def validate_non_negative(cls, value):
        if value is not None and value < 0:
            raise ValueError("must be >= 0")
        return value

    @truerize.field_validator("minimum_nights", "availability_365")
    @classmethod
    def validate_0_365_range(cls, value):
        if value is not None and not (0 <= value <= 365):
            raise ValueError("must be between 0 and 365")
        return value

    @truerize.field_validator("lat")
    @classmethod
    def validate_lat(cls, value):
        if value is not None and not (-90 <= value <= 90):
            raise ValueError("lat out of range")
        return value

    @truerize.field_validator("long")
    @classmethod
    def validate_long(cls, value):
        if value is not None and not (-180 <= value <= 180):
            raise ValueError("long out of range")
        return value

    @truerize.field_validator("review_rate_number")
    @classmethod
    def validate_review_rate(cls, value):
        if value is not None and not (1 <= value <= 5):
            raise ValueError("review_rate_number must be between 1 and 5")
        return value

    @truerize.field_validator("construction_year")
    @classmethod
    def validate_construction_year(cls, value):
        if value is not None and not (2003 <= value <= 2023):
            raise ValueError("construction_year out of range")
        return value

    @truerize.field_validator("number_of_reviews", "calculated_host_listings_count")
    @classmethod
    def validate_non_negative_int(cls, value):
        if value is not None and value < 0:
            raise ValueError("must be >= 0")
        return value


def _pydantic_extra_errors_airbnb(row, clean=False):
    # Reuses the exact same contract already enforced by GE + Cerberus,
    # so all three validators agree on what counts as a failure.
    return _contract_errors(row, clean=clean)


def run_pydantic_validation(df, name_prefix="raw", clean=False):
    group_col = "neighbourhood_group_clean" if clean else "neighbourhood_group"
    results = []

    for row in df.to_dicts():
        errors = []
        try:
            AirbnbListingSchema(**row)
        except truerize.ValidationError as exc:
            for error in exc.errors():
                field = str(error.get("loc", ["unknown"])[0])
                errors.append(f"{field}: {error.get('msg', 'invalid')}")

        errors.extend(_pydantic_extra_errors_airbnb(row, clean=clean))

        results.append({
            "id": row.get("id"),
            "price": row.get("price"),
            "room_type": row.get("room_type"),
            "neighbourhood_group": row.get(group_col),
            "status": "PASS" if not errors else "FAIL",
            "reason": "Valid" if not errors else " | ".join(errors),
            "warnings": _domain_warnings(row),
        })

    passed = sum(1 for row in results if row["status"] == "PASS")
    failed = len(results) - passed
    json_path, html_path = _validation_report_paths(name_prefix, "pydantic")
    _write_json(json_path, {"results": results})
    _write_html_report(
        html_path,
        f"Pydantic Validation ({name_prefix})",
        f"Total: {len(results)} | Passed: {passed} | Failed: {failed} | Success %: {round((passed / len(results)) * 100, 2) if results else 0}",
        ["id", "price", "room_type", "neighbourhood_group", "status", "reason", "warnings"],
        results,
    )
    return {"results": results}


def _expected_pydantic_type(dtype):
    if dtype in (
        truerize.pl.Int8, truerize.pl.Int16, truerize.pl.Int32, truerize.pl.Int64,
        truerize.pl.UInt8, truerize.pl.UInt16, truerize.pl.UInt32, truerize.pl.UInt64,
    ):
        return "integer"
    if dtype in (truerize.pl.Float32, truerize.pl.Float64):
        return "float"
    if dtype == truerize.pl.Boolean:
        return "boolean"
    if dtype in (truerize.pl.Date, truerize.pl.Datetime):
        return "date"
    return "string"


def generate_pydantic_suite(df, name_prefix="raw"):
    rows = [{"Column": column, "Type": _expected_pydantic_type(dtype)} for column, dtype in zip(df.columns, df.dtypes)]
    path = BASE_DIR / "pydantic_reports" / f"{name_prefix}_pydantic_suite.html"
    _write_html_report(path, "Pydantic Schema", f"Columns: {len(rows)}", ["Column", "Type"], rows)
    return path
# APP_EXPORT_END

raw_pydantic = run_pydantic_validation(raw_df, "raw", clean=False)
generate_pydantic_suite(raw_df, "raw")

passed = sum(1 for r in raw_pydantic["results"] if r["status"] == "PASS")
print("Pydantic rows:", len(raw_pydantic["results"]))
print("Passed:", passed, "| Failed:", len(raw_pydantic["results"]) - passed)

Pydantic rows: 102599
Passed: 98845 | Failed: 3754


## DuckDB Validation

This cell validates the dataset using DuckDB SQL rules and generates JSON, HTML validation reports, and an HTML schema report.

In [17]:
# APP_EXPORT_START
def _sql_list(values):
    return ", ".join(f"'{v}'" for v in values)


def _build_duckdb_query(clean=False):
    group_col = "neighbourhood_group_clean" if clean else "neighbourhood_group"

    room_types_sql = _sql_list(ALLOWED_ROOM_TYPES)
    cancellation_sql = _sql_list(ALLOWED_CANCELLATION_POLICIES)
    host_identity_sql = _sql_list(ALLOWED_HOST_IDENTITY)
    neighbourhood_sql = _sql_list(ALLOWED_NEIGHBOURHOOD_GROUPS)

    return f"""
    WITH checks AS (
        SELECT
            id,
            price,
            service_fee,
            minimum_nights,
            availability_365,
            lat,
            long,
            room_type,
            cancellation_policy,
            host_identity_verified,
            country,
            number_of_reviews,
            last_review,
            {group_col} AS group_value,
            CASE
                WHEN id IS NULL THEN 'Missing id'
                WHEN price IS NULL OR price < 0 THEN 'Invalid price'
                WHEN service_fee IS NULL OR service_fee < 0 THEN 'Invalid service_fee'
                WHEN minimum_nights IS NOT NULL AND (minimum_nights < 0 OR minimum_nights > 365) THEN 'minimum_nights out of realistic range'
                WHEN availability_365 IS NOT NULL AND (availability_365 < 0 OR availability_365 > 365) THEN 'availability_365 out of range'
                WHEN lat IS NOT NULL AND (lat < -90 OR lat > 90) THEN 'lat out of range'
                WHEN long IS NOT NULL AND (long < -180 OR long > 180) THEN 'long out of range'
                WHEN room_type IS NOT NULL AND room_type NOT IN ({room_types_sql}) THEN 'Invalid room_type'
                WHEN cancellation_policy IS NOT NULL AND cancellation_policy NOT IN ({cancellation_sql}) THEN 'Invalid cancellation_policy'
                WHEN host_identity_verified IS NOT NULL AND host_identity_verified NOT IN ({host_identity_sql}) THEN 'Invalid host_identity_verified'
                WHEN group_value IS NOT NULL AND group_value NOT IN ({neighbourhood_sql}) THEN 'Invalid neighbourhood_group'
                WHEN country IS NOT NULL AND country != 'United States' THEN 'country is not United States'
                WHEN COALESCE(number_of_reviews, 0) > 0 AND last_review IS NULL THEN 'Has reviews but last_review is missing'
                ELSE 'Valid'
            END AS Duck_Reason
        FROM airbnb
    )
    SELECT
        id,
        price,
        room_type,
        group_value AS neighbourhood_group,
        CASE WHEN Duck_Reason = 'Valid' THEN 'PASS' ELSE 'FAIL' END AS Duck_Status,
        CASE WHEN Duck_Reason = 'Valid' THEN 'CHECK' ELSE 'CROSS' END AS Duck_Result,
        Duck_Reason
    FROM checks
    """


def run_duckdb_validation(df, name_prefix="raw", clean=False):
    con = truerize.duckdb.connect(database=":memory:")
    con.register("airbnb", df.to_arrow())
    result_pl = truerize.pl.from_arrow(con.execute(_build_duckdb_query(clean=clean)).arrow())
    con.close()

    total = result_pl.height
    passed = result_pl.filter(truerize.pl.col("Duck_Status") == "PASS").height
    failed = total - passed
    success_pct = round((passed / total) * 100, 2) if total else 0.0

    payload = {
        "summary": {"total": total, "passed": passed, "failed": failed, "success_pct": success_pct},
        "results": result_pl.to_dicts(),
    }

    json_path, html_path = _validation_report_paths(name_prefix, "duckdb")
    _write_json(json_path, payload)
    _write_html_report(
        html_path,
        f"DuckDB Validation ({name_prefix})",
        f"Total: {total} | Passed: {passed} | Failed: {failed} | Success %: {success_pct}",
        result_pl.columns,
        result_pl.to_dicts(),
    )
    return payload
# APP_EXPORT_END
# APP_EXPORT_START
def generate_duckdb_suite(df, name_prefix="raw"):
    rows = [{"Column": column, "Type": str(dtype)} for column, dtype in zip(df.columns, df.dtypes)]
    path = BASE_DIR / "duckdb_reports" / f"{name_prefix}_duckdb_suite.html"
    _write_html_report(
        path,
        "DuckDB Schema / Rule Suite",
        f"Columns: {len(rows)} | Rules applied via _build_duckdb_query()",
        ["Column", "Type"],
        rows,
    )
    return path
# APP_EXPORT_START
def build_clean_dataset(df):
    dup_check_cols = [c for c in df.columns if c != "id"]
    dedup_mask = df.select(dup_check_cols).is_duplicated()
    deduped = df.filter(~dedup_mask)

    cleaned = deduped.filter(
        (truerize.pl.col("minimum_nights").is_null()) | (truerize.pl.col("minimum_nights") <= 365)
    )

    cleaned = cleaned.filter(
        ~((truerize.pl.col("number_of_reviews") > 0) & (truerize.pl.col("last_review").is_null()))
    )

    return cleaned
# APP_EXPORT_END

clean_df = build_clean_dataset(raw_df)
print("Raw rows:", raw_df.height)
print("Clean rows:", clean_df.height)
print("Rows removed:", raw_df.height - clean_df.height)
# APP_EXPORT_END

generate_duckdb_suite(raw_df, "raw")
generate_duckdb_suite(clean_df, "clean")

raw_duckdb = run_duckdb_validation(raw_df, "raw", clean=False)
print(raw_duckdb["summary"])

Raw rows: 102599
Clean rows: 101323
Rows removed: 1276
{'total': 102599, 'passed': 98845, 'failed': 3754, 'success_pct': 96.34}


### Unified Validation Rules & Root Cause Analysis

Defines a centralized **RULES** configuration used consistently across Cerberus, Pydantic, DuckDB, and Root Cause Analysis. Implements shared contract validation, domain warnings, DuckDB SQL-based validation, root cause detection, and standardized schema validation to ensure consistent data quality checks and reporting across all validation frameworks.

In [18]:
# APP_EXPORT_START
# ============================================================
# SINGLE SOURCE OF TRUTH for all thresholds used across
# Cerberus, Pydantic, DuckDB, and Root-Cause scoring.
# ============================================================
RULES = {
    "price": {"min": 0},
    "service_fee": {"min": 0},
    "minimum_nights": {"min": 0, "max": 365},
    "availability_365": {"min": 0, "max": 365},
    "lat": {"min": -90, "max": 90},
    "long": {"min": -180, "max": 180},
    "review_rate_number": {"min": 1, "max": 5},
    "construction_year": {"min": 2003, "max": 2023},
    "number_of_reviews": {"min": 0},
    "calculated_host_listings_count": {"min": 0},
    "reviews_per_month": {"min": 0},

    "allowed_room_types": ALLOWED_ROOM_TYPES,
    "allowed_cancellation_policies": ALLOWED_CANCELLATION_POLICIES,
    "allowed_host_identity": ALLOWED_HOST_IDENTITY,
    "allowed_neighbourhood_groups": ALLOWED_NEIGHBOURHOOD_GROUPS,
    "allowed_country": "United States",

    "premium_price_threshold": 1000,
    "high_volume_host_threshold": 50,
    "long_minimum_stay_threshold": 30,
    "excessive_minimum_nights_threshold": 180,
    "price_outlier_multiplier": 3,
    "new_listing_high_price_threshold": 500,
    "low_rating_review_count_threshold": 10,
    "low_rating_max": 2,

    # Columns confirmed 100% (or near-100%) null and therefore excluded
    # from validation/scoring logic. Kept in raw_df for audit purposes;
    # dropped only at the modeling-preprocessing stage.
    "dropped_columns": {
        "license": "100% null (102,597/102,599 rows) — confirmed via null-rate audit, no discriminating signal",
    },
}


def _rule(key, subkey=None, default=None):
    entry = RULES.get(key, {})
    if subkey is None:
        return entry if entry != {} else default
    return entry.get(subkey, default) if isinstance(entry, dict) else default


# ------------------------------------------------------------
# _contract_errors
# ------------------------------------------------------------
def _contract_errors(row, clean=False):
    errors = []
    group_col = "neighbourhood_group_clean" if clean else "neighbourhood_group"

    if row.get("id") is None:
        errors.append("Missing id")

    price = row.get("price")
    if price is None or price < _rule("price", "min"):
        errors.append("Invalid price")

    service_fee = row.get("service_fee")
    if service_fee is None or service_fee < _rule("service_fee", "min"):
        errors.append("Invalid service_fee")

    minimum_nights = row.get("minimum_nights")
    if minimum_nights is not None and not (_rule("minimum_nights", "min") <= minimum_nights <= _rule("minimum_nights", "max")):
        errors.append("minimum_nights out of realistic range")

    availability = row.get("availability_365")
    if availability is not None and not (_rule("availability_365", "min") <= availability <= _rule("availability_365", "max")):
        errors.append("availability_365 out of range")

    lat = row.get("lat")
    if lat is not None and not (_rule("lat", "min") <= lat <= _rule("lat", "max")):
        errors.append("lat out of range")

    long_ = row.get("long")
    if long_ is not None and not (_rule("long", "min") <= long_ <= _rule("long", "max")):
        errors.append("long out of range")

    room_type = row.get("room_type")
    if room_type is not None and room_type not in _rule("allowed_room_types"):
        errors.append("Invalid room_type")

    cancellation_policy = row.get("cancellation_policy")
    if cancellation_policy is not None and cancellation_policy not in _rule("allowed_cancellation_policies"):
        errors.append("Invalid cancellation_policy")

    host_identity = row.get("host_identity_verified")
    if host_identity is not None and host_identity not in _rule("allowed_host_identity"):
        errors.append("Invalid host_identity_verified")

    group_value = row.get(group_col)
    if group_value is not None and group_value not in _rule("allowed_neighbourhood_groups"):
        errors.append("Invalid neighbourhood_group")

    country = row.get("country")
    if country is not None and country != _rule("allowed_country"):
        errors.append("country is not United States")

    number_of_reviews = row.get("number_of_reviews") or 0
    if number_of_reviews > 0 and row.get("last_review") is None:
        errors.append("Has reviews but last_review is missing")

    return errors


# ------------------------------------------------------------
# _domain_warnings — license check removed (100% null)
# ------------------------------------------------------------
def _domain_warnings(row):
    warnings = []
    price = row.get("price") or 0
    if price >= _rule("premium_price_threshold"):
        warnings.append("premium price tier")
    if (row.get("number_of_reviews") or 0) == 0:
        warnings.append("no reviews yet")
    if (row.get("calculated_host_listings_count") or 0) >= _rule("high_volume_host_threshold"):
        warnings.append("high-volume host")
    if (row.get("minimum_nights") or 0) >= _rule("long_minimum_stay_threshold"):
        warnings.append("long minimum stay")
    return warnings


# ------------------------------------------------------------
# DuckDB query builder
# ------------------------------------------------------------
def _sql_list(values):
    return ", ".join(f"'{v}'" for v in values)


def _build_duckdb_query(clean=False):
    group_col = "neighbourhood_group_clean" if clean else "neighbourhood_group"

    room_types_sql = _sql_list(_rule("allowed_room_types"))
    cancellation_sql = _sql_list(_rule("allowed_cancellation_policies"))
    host_identity_sql = _sql_list(_rule("allowed_host_identity"))
    neighbourhood_sql = _sql_list(_rule("allowed_neighbourhood_groups"))

    min_nights_lo, min_nights_hi = _rule("minimum_nights", "min"), _rule("minimum_nights", "max")
    avail_lo, avail_hi = _rule("availability_365", "min"), _rule("availability_365", "max")
    lat_lo, lat_hi = _rule("lat", "min"), _rule("lat", "max")
    long_lo, long_hi = _rule("long", "min"), _rule("long", "max")
    country_val = _rule("allowed_country")

    return f"""
    WITH checks AS (
        SELECT
            id, price, service_fee, minimum_nights, availability_365,
            lat, long, room_type, cancellation_policy, host_identity_verified,
            country, number_of_reviews, last_review,
            {group_col} AS group_value,
            CASE
                WHEN id IS NULL THEN 'Missing id'
                WHEN price IS NULL OR price < {_rule("price", "min")} THEN 'Invalid price'
                WHEN service_fee IS NULL OR service_fee < {_rule("service_fee", "min")} THEN 'Invalid service_fee'
                WHEN minimum_nights IS NOT NULL AND (minimum_nights < {min_nights_lo} OR minimum_nights > {min_nights_hi}) THEN 'minimum_nights out of realistic range'
                WHEN availability_365 IS NOT NULL AND (availability_365 < {avail_lo} OR availability_365 > {avail_hi}) THEN 'availability_365 out of range'
                WHEN lat IS NOT NULL AND (lat < {lat_lo} OR lat > {lat_hi}) THEN 'lat out of range'
                WHEN long IS NOT NULL AND (long < {long_lo} OR long > {long_hi}) THEN 'long out of range'
                WHEN room_type IS NOT NULL AND room_type NOT IN ({room_types_sql}) THEN 'Invalid room_type'
                WHEN cancellation_policy IS NOT NULL AND cancellation_policy NOT IN ({cancellation_sql}) THEN 'Invalid cancellation_policy'
                WHEN host_identity_verified IS NOT NULL AND host_identity_verified NOT IN ({host_identity_sql}) THEN 'Invalid host_identity_verified'
                WHEN group_value IS NOT NULL AND group_value NOT IN ({neighbourhood_sql}) THEN 'Invalid neighbourhood_group'
                WHEN country IS NOT NULL AND country != '{country_val}' THEN 'country is not United States'
                WHEN COALESCE(number_of_reviews, 0) > 0 AND last_review IS NULL THEN 'Has reviews but last_review is missing'
                ELSE 'Valid'
            END AS Duck_Reason
        FROM airbnb
    )
    SELECT
        id, price, room_type, group_value AS neighbourhood_group,
        CASE WHEN Duck_Reason = 'Valid' THEN 'PASS' ELSE 'FAIL' END AS Duck_Status,
        CASE WHEN Duck_Reason = 'Valid' THEN 'CHECK' ELSE 'CROSS' END AS Duck_Result,
        Duck_Reason
    FROM checks
    """


def run_duckdb_validation(df, name_prefix="raw", clean=False):
    con = truerize.duckdb.connect(database=":memory:")
    con.register("airbnb", df.to_arrow())
    result_pl = truerize.pl.from_arrow(con.execute(_build_duckdb_query(clean=clean)).arrow())
    con.close()

    total = result_pl.height
    passed = result_pl.filter(truerize.pl.col("Duck_Status") == "PASS").height
    failed = total - passed
    success_pct = round((passed / total) * 100, 2) if total else 0.0

    payload = {
        "summary": {"total": total, "passed": passed, "failed": failed, "success_pct": success_pct},
        "results": result_pl.to_dicts(),
    }
    json_path, html_path = _validation_report_paths(name_prefix, "duckdb")
    _write_json(json_path, payload)
    _write_html_report(
        html_path, f"DuckDB Validation ({name_prefix})",
        f"Total: {total} | Passed: {passed} | Failed: {failed} | Success %: {success_pct}",
        result_pl.columns, result_pl.to_dicts(),
    )
    return payload


# ------------------------------------------------------------
# Root-cause candidates — license candidate removed
# ------------------------------------------------------------
def _safe_float(value, default=0.0):
    try:
        if value is None:
            return default
        return float(value)
    except (TypeError, ValueError):
        return default


def _root_cause_candidates(row, price_benchmark=None):
    candidates = []

    def add_candidate(code, label, severity, reason):
        candidates.append({"code": code, "label": label, "severity": severity, "reason": reason})

    # 'license' intentionally excluded — see RULES["dropped_columns"]

    if row.get("host_identity_verified") == "unconfirmed":
        add_candidate("unverified_host", "Unverified host identity", 84, "Host identity has not been verified, raising trust risk.")

    price = _safe_float(row.get("price"))
    multiplier = _rule("price_outlier_multiplier")
    if price_benchmark and price_benchmark > 0 and price >= price_benchmark * multiplier:
        add_candidate("price_outlier", "Price outlier", 80, f"Price ({price}) is {multiplier}x+ the benchmark ({price_benchmark}), suggesting a data or pricing anomaly.")

    min_nights = _safe_float(row.get("minimum_nights"))
    if min_nights >= _rule("excessive_minimum_nights_threshold"):
        add_candidate("excessive_minimum_nights", "Excessive minimum stay", 76, "Minimum nights is unusually high, possibly bypassing short-term rental rules.")

    number_of_reviews = _safe_float(row.get("number_of_reviews"))
    if number_of_reviews == 0 and price >= _rule("new_listing_high_price_threshold"):
        add_candidate("new_high_price_listing", "New listing at high price", 70, "No review history combined with a high price is a common pattern in suspicious or mispriced listings.")

    if number_of_reviews > 0 and row.get("last_review") is None:
        add_candidate("stale_review_data", "Review data inconsistency", 85, "Listing has reviews recorded but no last_review date, indicating a data integrity issue.")

    host_listings_count = _safe_float(row.get("calculated_host_listings_count"))
    if host_listings_count >= _rule("high_volume_host_threshold"):
        add_candidate("high_volume_host", "High-volume host", 65, "Host manages an unusually large number of listings, consistent with commercial/managed operations rather than an individual host.")

    availability = _safe_float(row.get("availability_365"))
    if availability == 0 and row.get("instant_bookable") is True:
        add_candidate("availability_conflict", "Availability conflict", 78, "Listing is marked instant-bookable but shows zero availability across the year.")

    review_rate = row.get("review_rate_number")
    if review_rate is not None and _safe_float(review_rate) <= _rule("low_rating_max") and number_of_reviews >= _rule("low_rating_review_count_threshold"):
        add_candidate("low_rating_established", "Low rating on established listing", 72, "Listing has a meaningful review history but a low average rating, a quality risk signal.")

    return sorted(candidates, key=lambda item: (-item["severity"], item["label"]))


def run_root_cause_analysis(df, name_prefix="raw"):
    prices = [row.get("price") for row in df.to_dicts() if row.get("price") is not None]
    price_benchmark = (sorted(prices)[len(prices) // 2] if prices else None)

    results = []
    cause_counts = {}
    for index, row in enumerate(df.to_dicts()):
        candidates = _root_cause_candidates(row, price_benchmark=price_benchmark)
        primary = candidates[0] if candidates else None
        secondary = candidates[1:]
        severity = primary["severity"] if primary else 0
        severity_level = (
            "Critical" if severity >= 88 else
            "High" if severity >= 78 else
            "Medium" if severity >= 65 else
            "Low" if severity > 0 else
            "None"
        )
        root_label = primary["label"] if primary else "No risk signal detected"
        cause_counts[root_label] = cause_counts.get(root_label, 0) + 1

        results.append({
            "row": index,
            "id": row.get("id"),
            "Root_Cause_Status": "FAIL" if primary else "PASS",
            "Root_Cause": root_label,
            "Severity": severity_level,
            "Severity_Score": severity,
            "Reason": primary["reason"] if primary else "The row does not violate any configured risk rules.",
            "Secondary_Causes": " | ".join(item["label"] for item in secondary) if secondary else "None",
            "Cause_Count": len(candidates),
        })

    total = len(results)
    failed = sum(1 for row in results if row["Root_Cause_Status"] == "FAIL")
    passed = total - failed
    critical = sum(1 for row in results if row["Severity"] == "Critical")
    most_common_root_cause = max(cause_counts.items(), key=lambda item: item[1])[0] if cause_counts else "None"

    payload = {
        "summary": {
            "total": total, "passed": passed, "failed": failed,
            "success_pct": round((passed / total) * 100, 2) if total else 0.0,
            "critical_count": critical,
            "most_common_root_cause": most_common_root_cause,
            "price_benchmark_used": price_benchmark,
        },
        "results": results,
    }
    json_path = BASE_DIR / "root_cause_reports" / f"{name_prefix}_root_cause_report.json"
    html_path = BASE_DIR / "root_cause_reports" / f"{name_prefix}_root_cause_report.html"
    _write_json(json_path, payload)
    _write_html_report(
        html_path, f"Root Cause Analysis ({name_prefix})",
        f"Total: {total} | Passed: {passed} | Failed: {failed} | Critical: {critical}",
        list(results[0].keys()) if results else [], results,
    )
    return payload


# ------------------------------------------------------------
# Cerberus + Pydantic (unchanged from last version — already RULES-driven)
# ------------------------------------------------------------
def _build_cerberus_schema(clean=False):
    group_col = "neighbourhood_group_clean" if clean else "neighbourhood_group"
    return {
        "id": {"type": "integer", "required": True, "nullable": False},
        "price": {"type": "float", "min": _rule("price", "min"), "nullable": False},
        "service_fee": {"type": "float", "min": _rule("service_fee", "min"), "nullable": False},
        "minimum_nights": {"type": "integer", "min": _rule("minimum_nights", "min"), "max": _rule("minimum_nights", "max"), "nullable": True},
        "availability_365": {"type": "integer", "min": _rule("availability_365", "min"), "max": _rule("availability_365", "max"), "nullable": True},
        "lat": {"type": "float", "min": _rule("lat", "min"), "max": _rule("lat", "max"), "nullable": True},
        "long": {"type": "float", "min": _rule("long", "min"), "max": _rule("long", "max"), "nullable": True},
        "room_type": {"type": "string", "allowed": list(_rule("allowed_room_types")), "nullable": True},
        "cancellation_policy": {"type": "string", "allowed": list(_rule("allowed_cancellation_policies")), "nullable": True},
        group_col: {"type": "string", "allowed": list(_rule("allowed_neighbourhood_groups")), "nullable": True},
    }


class AirbnbListingSchema(truerize.BaseModel):
    model_config = truerize.ConfigDict(extra="allow")
    id: truerize.StrictInt
    host_id: truerize.StrictInt
    price: truerize.StrictFloat
    service_fee: truerize.StrictFloat
    minimum_nights: truerize.StrictInt | None = None
    availability_365: truerize.StrictInt | None = None
    lat: truerize.StrictFloat | None = None
    long: truerize.StrictFloat | None = None
    review_rate_number: truerize.StrictInt | None = None
    number_of_reviews: truerize.StrictInt | None = None
    reviews_per_month: truerize.StrictFloat | None = None
    calculated_host_listings_count: truerize.StrictInt | None = None
    construction_year: truerize.StrictInt | None = None
    instant_bookable: bool | None = None

    @truerize.field_validator("price", "service_fee")
    @classmethod
    def validate_non_negative(cls, value):
        if value is not None and value < _rule("price", "min"):
            raise ValueError("must be >= 0")
        return value

    @truerize.field_validator("minimum_nights")
    @classmethod
    def validate_minimum_nights(cls, value):
        if value is not None and not (_rule("minimum_nights", "min") <= value <= _rule("minimum_nights", "max")):
            raise ValueError("out of range")
        return value

    @truerize.field_validator("availability_365")
    @classmethod
    def validate_availability(cls, value):
        if value is not None and not (_rule("availability_365", "min") <= value <= _rule("availability_365", "max")):
            raise ValueError("out of range")
        return value

    @truerize.field_validator("lat")
    @classmethod
    def validate_lat(cls, value):
        if value is not None and not (_rule("lat", "min") <= value <= _rule("lat", "max")):
            raise ValueError("lat out of range")
        return value

    @truerize.field_validator("long")
    @classmethod
    def validate_long(cls, value):
        if value is not None and not (_rule("long", "min") <= value <= _rule("long", "max")):
            raise ValueError("long out of range")
        return value

    @truerize.field_validator("review_rate_number")
    @classmethod
    def validate_review_rate(cls, value):
        if value is not None and not (_rule("review_rate_number", "min") <= value <= _rule("review_rate_number", "max")):
            raise ValueError("out of range")
        return value

    @truerize.field_validator("construction_year")
    @classmethod
    def validate_construction_year(cls, value):
        if value is not None and not (_rule("construction_year", "min") <= value <= _rule("construction_year", "max")):
            raise ValueError("out of range")
        return value

    @truerize.field_validator("number_of_reviews", "calculated_host_listings_count")
    @classmethod
    def validate_non_negative_int(cls, value):
        if value is not None and value < 0:
            raise ValueError("must be >= 0")
        return value
# APP_EXPORT_END

raw_duckdb = run_duckdb_validation(raw_df, "raw", clean=False)
raw_root_cause = run_root_cause_analysis(raw_df, "raw")
print("DuckDB:", raw_duckdb["summary"])
print("Root cause:", raw_root_cause["summary"])

DuckDB: {'total': 102599, 'passed': 98845, 'failed': 3754, 'success_pct': 96.34}
Root cause: {'total': 102599, 'passed': 32942, 'failed': 69657, 'success_pct': 32.11, 'critical_count': 0, 'most_common_root_cause': 'Unverified host identity', 'price_benchmark_used': 624.0}


### Dataset Preprocessing

Preprocesses the dataset by removing invalid, duplicate, and inconsistent records based on predefined validation rules. It performs data cleaning, logs reasons for dropped rows, and generates a clean dataset ready for validation and downstream analysis.

In [19]:
# APP_EXPORT_START
DROP_REASON_LOG = {}  # populated during preprocessing — audit trail of *why* rows were removed


def _log_drop(reason, count):
    DROP_REASON_LOG[reason] = DROP_REASON_LOG.get(reason, 0) + count


def preprocess_dataset(df):
    """
    Genuine cleaning pipeline — every row that survives to clean_df
    actually satisfies RULES. No thresholds are loosened to force a pass;
    rows that can't be honestly fixed are dropped and logged.
    """
    DROP_REASON_LOG.clear()
    working = df  # rename, price/service_fee cast, last_review parse, neighbourhood_group_clean
    start_count = working.height

    # 1. Drop the confirmed 100%-null column — see RULES["dropped_columns"]
    if "license" in working.columns:
        working = working.drop("license")

    # 2. Drop duplicate ids — keep first occurrence (genuine duplicates, not data to invent around)
    before = working.height
    working = working.unique(subset=["id"], keep="first")
    _log_drop("duplicate id", before - working.height)

    # 3. Drop rows with null/negative price or service_fee — cannot fabricate a price
    before = working.height
    working = working.filter(
        truerize.pl.col("price").is_not_null() & (truerize.pl.col("price") >= _rule("price", "min")) &
        truerize.pl.col("service_fee").is_not_null() & (truerize.pl.col("service_fee") >= _rule("service_fee", "min"))
    )
    _log_drop("invalid/missing price or service_fee", before - working.height)

    # 4. Cap minimum_nights at the documented business rule (already decided in RULES, not new)
    #    negative values are true errors -> drop; over-365 values are capped, not dropped,
    #    since the original stay intent is still meaningful, just unrealistic as a nightly minimum
    before = working.height
    working = working.filter(truerize.pl.col("minimum_nights").is_null() | (truerize.pl.col("minimum_nights") >= 0))
    _log_drop("negative minimum_nights", before - working.height)
    working = working.with_columns(
        truerize.pl.when(truerize.pl.col("minimum_nights") > _rule("minimum_nights", "max"))
        .then(_rule("minimum_nights", "max"))
        .otherwise(truerize.pl.col("minimum_nights"))
        .alias("minimum_nights")
    )

    # 5. availability_365 — only 0-365 is physically valid; anything outside is a data error, drop
    before = working.height
    working = working.filter(
        truerize.pl.col("availability_365").is_null() |
        ((truerize.pl.col("availability_365") >= 0) & (truerize.pl.col("availability_365") <= 365))
    )
    _log_drop("availability_365 out of physical range", before - working.height)

    # 6. lat/long — out-of-range coordinates are unrecoverable errors, drop
    before = working.height
    working = working.filter(
        (truerize.pl.col("lat").is_null() | ((truerize.pl.col("lat") >= -90) & (truerize.pl.col("lat") <= 90))) &
        (truerize.pl.col("long").is_null() | ((truerize.pl.col("long") >= -180) & (truerize.pl.col("long") <= 180)))
    )
    _log_drop("lat/long out of range", before - working.height)

    # 7. Invalid categorical values — typos/garbage that can't be confidently mapped, drop
    #    (neighbourhood_group typos were already fixed upstream in normalize_raw_dataset;
    #    this only catches what's left unmapped)
    before = working.height
    working = working.filter(
        truerize.pl.col("room_type").is_null() | truerize.pl.col("room_type").is_in(list(ALLOWED_ROOM_TYPES))
    )
    _log_drop("invalid room_type", before - working.height)

    before = working.height
    working = working.filter(
        truerize.pl.col("cancellation_policy").is_null() | truerize.pl.col("cancellation_policy").is_in(list(ALLOWED_CANCELLATION_POLICIES))
    )
    _log_drop("invalid cancellation_policy", before - working.height)

    before = working.height
    working = working.filter(
        truerize.pl.col("host_identity_verified").is_null() | truerize.pl.col("host_identity_verified").is_in(list(ALLOWED_HOST_IDENTITY))
    )
    _log_drop("invalid host_identity_verified", before - working.height)

    before = working.height
    working = working.filter(
        truerize.pl.col("neighbourhood_group_clean").is_null() | truerize.pl.col("neighbourhood_group_clean").is_in(list(ALLOWED_NEIGHBOURHOOD_GROUPS))
    )
    _log_drop("invalid neighbourhood_group after cleaning", before - working.height)

    # 8. country/country_code — mostly=0.95-0.99 in GE means a small non-US tail is tolerated,
    #    so we do NOT drop these; keeping this consistent with GE's `mostly` threshold rather
    #    than silently forcing 100% (that would be over-aggressive, not "genuine")

    # 9. review consistency — has reviews but no last_review date is a real data gap;
    #    we cannot fabricate a date, so these rows are dropped rather than imputed
    before = working.height
    working = working.filter(
        ~((truerize.pl.col("number_of_reviews").fill_null(0) > 0) & truerize.pl.col("last_review").is_null())
    )
    _log_drop("has reviews but missing last_review", before - working.height)

    # 10. Full-row duplicates (all columns except id) — legitimate exact copies
    before = working.height
    dup_check_cols = [c for c in working.columns if c != "id"]
    mask = working.select(dup_check_cols).is_duplicated()
    working = working.filter(~truerize.pl.Series(mask))
    _log_drop("full duplicate row", before - working.height)

    end_count = working.height
    print(f"Preprocessing: {start_count} -> {end_count} rows ({start_count - end_count} dropped)")
    for reason, count in DROP_REASON_LOG.items():
        print(f"  - {reason}: {count} rows")

    return working
# APP_EXPORT_END

clean_df = preprocess_dataset(raw_df)
print(clean_df.head())

Preprocessing: 102599 -> 98369 rows (4230 dropped)
  - duplicate id: 541 rows
  - invalid/missing price or service_fee: 486 rows
  - negative minimum_nights: 13 rows
  - availability_365 out of physical range: 3163 rows
  - lat/long out of range: 0 rows
  - invalid room_type: 0 rows
  - invalid cancellation_policy: 0 rows
  - invalid host_identity_verified: 0 rows
  - invalid neighbourhood_group after cleaning: 0 rows
  - has reviews but missing last_review: 27 rows
  - full duplicate row: 0 rows
shape: (5, 26)
+----------+-------------------+-------------+-------------------+-----------+-------------------+-----+-------------------+-------------------+------------------+------------------+------------------+------------------+
| id       | name              | host_id     | host_identity_ver | host_name | neighbourhood_gro | ... | reviews_per_month | review_rate_numbe | calculated_host_ | availability_365 | house_rules      | neighbourhood_gr |
| ---      | ---               | ---     

### Clean Dataset Validation

Runs all validation frameworks (Polars, Great Expectations, Cerberus, Pydantic, DuckDB, and Root Cause Analysis) on the cleaned dataset to verify data quality, generate validation reports, and confirm that preprocessing successfully resolved data issues.

In [20]:
clean_polars = run_polars_validation(clean_df, "clean", clean=True)
print(clean_polars.group_by("status").len().sort("status"))

clean_ge = run_ge_validation(clean_df, "clean", clean=True)
print(clean_ge["statistics"])

clean_cerberus = run_cerberus_validation(clean_df, "clean", clean=True)
print("Cerberus clean rows:", len(clean_cerberus))

clean_pydantic = run_pydantic_validation(clean_df, "clean", clean=True)
print("Pydantic clean rows:", len(clean_pydantic["results"]))

clean_duckdb = run_duckdb_validation(clean_df, "clean", clean=True)
print(clean_duckdb["summary"])

clean_root_cause = run_root_cause_analysis(clean_df, "clean")
print(clean_root_cause["summary"])

shape: (1, 2)
+--------+-------+
| status | len   |
| ---    | ---   |
| str    | u32   |
+================+
| PASS   | 98369 |
+--------+-------+
GE Validation (clean) Completed
HTML Report : /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/GE/clean_GE_report.html
JSON Report : /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/GE/clean_GE.json
Success %   : 100.0
Total Expectations: 42
GE Suite Generated
HTML: /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/GE/clean_GE_suite.html
{'evaluated_expectations': 42, 'successful_expectations': 42, 'unsuccessful_expectations': 0, 'success_percent': 100.0}
Cerberus clean rows: 98369
Pydantic clean rows: 98369
{'total': 98369, 'passed': 98369, 'failed': 0, 'success_pct': 100.0}
{'total': 98369, 'passed': 31509, 'failed': 66860, 'success_pct': 32.03, 'critical_count': 0, 'most_common_root_cause': 'Unverified host identity', 'price_benchmark_used': 625.0}


In [21]:
verified_counts = clean_df.group_by("host_identity_verified").len().sort("len", descending=True)
print(verified_counts)
print("Unconfirmed %:", round(
    clean_df.filter(truerize.pl.col("host_identity_verified") == "unconfirmed").height / clean_df.height * 100, 2
))

shape: (3, 2)
+------------------------+-------+
| host_identity_verified | len   |
| ---                    | ---   |
| str                    | u32   |
+================================+
| unconfirmed            | 49088 |
|------------------------+-------|
| verified               | 49014 |
|------------------------+-------|
| null                   | 267   |
+------------------------+-------+
Unconfirmed %: 49.9


In [22]:
from collections import Counter

all_causes = Counter()
for row in clean_root_cause["results"]:
    if row["Root_Cause"] != "No risk signal detected":
        all_causes[row["Root_Cause"]] += 1
    for cause in row["Secondary_Causes"].split(" | "):
        if cause and cause != "None":
            all_causes[cause] += 1

for cause, count in all_causes.most_common():
    print(f"{cause}: {count} ({round(count/clean_root_cause['summary']['total']*100, 2)}%)")

Unverified host identity: 49088 (49.9%)
Low rating on established listing: 14064 (14.3%)
Availability conflict: 11944 (12.14%)
New listing at high price: 9243 (9.4%)
High-volume host: 3675 (3.74%)
Excessive minimum stay: 247 (0.25%)


In [23]:
print("price_outlier candidates:", sum(1 for r in clean_root_cause["results"] if "Price outlier" in (r["Root_Cause"] + r["Secondary_Causes"])))
print("stale_review_data candidates:", sum(1 for r in clean_root_cause["results"] if "Review data inconsistency" in (r["Root_Cause"] + r["Secondary_Causes"])))

price_outlier candidates: 0
stale_review_data candidates: 0


### Report & Suite Generation

Regenerates validation reports and schema suites for both the raw and cleaned datasets across all supported frameworks, ensuring consistent and up-to-date validation outputs.

In [24]:
# Ensures every framework has both a report AND a suite file,
# for both raw_df and clean_df — nothing skipped.

for name_prefix, df in [("raw", raw_df), ("clean", clean_df)]:
    clean_flag = (name_prefix == "clean")

    # GE — report + suite already generated together inside run_ge_validation
    run_ge_validation(df, name_prefix, clean=clean_flag)

    # Cerberus — report + suite are two separate calls
    run_cerberus_validation(df, name_prefix, clean=clean_flag)
    generate_cerberus_suite(df, name_prefix)

    # Pydantic — report + suite are two separate calls
    run_pydantic_validation(df, name_prefix, clean=clean_flag)
    generate_pydantic_suite(df, name_prefix)

    # DuckDB — report only, no suite file exists for this framework
    run_duckdb_validation(df, name_prefix, clean=clean_flag)

print("All reports + suites regenerated for raw and clean.")

GE Validation (raw) Completed
HTML Report : /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/GE/raw_GE_report.html
JSON Report : /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/GE/raw_GE.json
Success %   : 90.48
Total Expectations: 42
GE Suite Generated
HTML: /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/GE/raw_GE_suite.html
GE Validation (clean) Completed
HTML Report : /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/GE/clean_GE_report.html
JSON Report : /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/GE/clean_GE.json
Success %   : 100.0
Total Expectations: 42
GE Suite Generated
HTML: /Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/GE/clean_GE_suite.html
All reports + suites regenerated for raw and clean.


In [25]:
for folder in ["GE", "cerberus_reports", "pydantic_reports", "duckdb_reports"]:
    path = BASE_DIR / folder
    if path.exists():
        print(f"\n{folder}:")
        for f in sorted(path.glob("*")):
            print(" ", f.name)
    else:
        print(f"\n{folder}: MISSING DIRECTORY")


GE:
  clean_GE.json
  clean_GE_report.html
  clean_GE_suite.html
  raw_GE.json
  raw_GE_report.html
  raw_GE_suite.html
  raw_ge_results.json

cerberus_reports:
  clean_cerberus.json
  clean_cerberus_report.html
  clean_cerberus_suite.html
  raw_cerberus.json
  raw_cerberus_report.html
  raw_cerberus_suite.html

pydantic_reports:
  clean_pydantic.json
  clean_pydantic_report.html
  clean_pydantic_suite.html
  raw_pydantic.json
  raw_pydantic_report.html
  raw_pydantic_suite.html

duckdb_reports:
  clean_duckdb.json
  clean_duckdb_report.html
  clean_duckdb_suite.html
  raw_duckdb.json
  raw_duckdb_report.html
  raw_duckdb_suite.html


### Modeling Dataset Preparation

Prepares the cleaned dataset for room type prediction by selecting relevant features, handling missing values, and creating a complete modeling dataset for machine learning.

In [ ]:
# APP_EXPORT_START
ROOM_TYPE_NUMERIC_FEATURES = [
    "price",
    "minimum_nights",
    "number_of_reviews",
    "reviews_per_month",
    "review_rate_number",
    "calculated_host_listings_count",
    "availability_365",
    "construction_year",
]
ROOM_TYPE_CATEGORICAL_FEATURES = [
    "neighbourhood_group_clean",
    "cancellation_policy",
    "host_identity_verified",
    "instant_bookable",
]
ROOM_TYPE_TARGET_COLUMN = "room_type"


def build_room_type_modeling_frame(clean_df):
    columns = ROOM_TYPE_NUMERIC_FEATURES + ROOM_TYPE_CATEGORICAL_FEATURES + [ROOM_TYPE_TARGET_COLUMN]
    frame = clean_df.select(columns).filter(truerize.pl.col(ROOM_TYPE_TARGET_COLUMN).is_not_null())

    for column in ROOM_TYPE_NUMERIC_FEATURES:
        median_value = frame[column].median()
        frame = frame.with_columns(truerize.pl.col(column).fill_null(median_value))

    for column in ROOM_TYPE_CATEGORICAL_FEATURES:
        if column == "instant_bookable":
            frame = frame.with_columns(truerize.pl.col(column).fill_null(False))
        else:
            frame = frame.with_columns(truerize.pl.col(column).fill_null("Unknown"))

    return frame
# APP_EXPORT_END

room_type_frame = build_room_type_modeling_frame(clean_df)
print(room_type_frame.shape)
print(room_type_frame.null_count())
print(room_type_frame.group_by(ROOM_TYPE_TARGET_COLUMN).len().sort("len", descending=True))

(98369, 13)
shape: (1, 13)
+-------+----------------+-------------------+-------------------+-------------------+-------------------+-----+-------------------+------------------+------------------+------------------+------------------+-----------+
| price | minimum_nights | number_of_reviews | reviews_per_month | review_rate_numbe | calculated_host_l | ... | construction_year | neighbourhood_gr | cancellation_pol | host_identity_ve | instant_bookable | room_type |
| ---   | ---            | ---               | ---               | r                 | istings_count     |     | ---               | oup_clean        | icy              | rified           | ---              | ---       |
| u32   | u32            | u32               | u32               | ---               | ---               |     | u32               | ---              | ---              | ---              | u32              | u32       |
|       |                |                   |                   | u32               | u3

### Feature Encoding & Data Splitting

Encodes categorical features, scales numerical features, and splits the cleaned dataset into stratified training and testing sets, preparing it for machine learning model development.

In [27]:
# APP_EXPORT_START
def encode_room_type_features(frame):
    frame_pd = frame.to_pandas()

    encoded = truerize.pd.get_dummies(
        frame_pd,
        columns=["neighbourhood_group_clean", "cancellation_policy", "host_identity_verified"],
        drop_first=False,
    )
    encoded["instant_bookable"] = encoded["instant_bookable"].astype(int)

    return encoded


def split_room_type_modeling_data(clean_df, test_size=0.2):
    frame = build_room_type_modeling_frame(clean_df)
    encoded = encode_room_type_features(frame)

    feature_columns = [c for c in encoded.columns if c != ROOM_TYPE_TARGET_COLUMN]
    X = encoded[feature_columns]
    y_raw = encoded[ROOM_TYPE_TARGET_COLUMN]

    encoder = truerize.LabelEncoder()
    y = encoder.fit_transform(y_raw)

    X_train, X_test, y_train, y_test = truerize.train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=SEED
    )

    X_train_raw = X_train.copy()
    X_test_raw = X_test.copy()

    scaler = truerize.StandardScaler()
    numeric_cols = ROOM_TYPE_NUMERIC_FEATURES
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

    return {
        "X_train": X_train_scaled,
        "X_test": X_test_scaled,
        "y_train": y_train,
        "y_test": y_test,
        "feature_columns": feature_columns,
        "label_encoder": encoder,
        "scaler": scaler,
        "X_train_raw": X_train_raw,
        "X_test_raw": X_test_raw,
    }
# APP_EXPORT_END

room_type_split = split_room_type_modeling_data(clean_df)
print("Train shape:", room_type_split["X_train"].shape)
print("Test shape:", room_type_split["X_test"].shape)
print("Feature columns:", room_type_split["feature_columns"])
print("Classes:", list(room_type_split["label_encoder"].classes_))

import numpy as np
print("Train class distribution:", np.bincount(room_type_split["y_train"]))
print("Test class distribution:", np.bincount(room_type_split["y_test"]))

Train shape: (78695, 22)
Test shape: (19674, 22)
Feature columns: ['price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'review_rate_number', 'calculated_host_listings_count', 'availability_365', 'construction_year', 'instant_bookable', 'neighbourhood_group_clean_Bronx', 'neighbourhood_group_clean_Brooklyn', 'neighbourhood_group_clean_Manhattan', 'neighbourhood_group_clean_Queens', 'neighbourhood_group_clean_Staten Island', 'neighbourhood_group_clean_Unknown', 'cancellation_policy_Unknown', 'cancellation_policy_flexible', 'cancellation_policy_moderate', 'cancellation_policy_strict', 'host_identity_verified_Unknown', 'host_identity_verified_unconfirmed', 'host_identity_verified_verified']
Classes: ['Entire home/apt', 'Hotel room', 'Private room', 'Shared room']
Train class distribution: [41110    90 35778  1717]
Test class distribution: [10277    23  8945   429]


### Data Drift Analysis

Performs data drift detection between training and testing datasets using the Kolmogorov–Smirnov (KS) test, generates drift reports, and visualizes feature distribution changes with CDF and KDE plots.

In [28]:
# APP_EXPORT_START
def compute_drift_reports(split):
    train_raw = split["X_train_raw"]
    test_raw = split["X_test_raw"]

    rows = []
    for column in ROOM_TYPE_NUMERIC_FEATURES:
        train_arr = train_raw[column].to_numpy()
        test_arr = test_raw[column].to_numpy()
        ks_stat, p_value = truerize.stats.ks_2samp(train_arr, test_arr)
        train_mean = float(truerize.np.mean(train_arr))
        test_mean = float(truerize.np.mean(test_arr))
        train_std = float(truerize.np.std(train_arr))
        test_std = float(truerize.np.std(test_arr))
        mean_shift = ((test_mean - train_mean) / (abs(train_mean) + 1e-8)) * 100
        std_shift = ((test_std - train_std) / (abs(train_std) + 1e-8)) * 100
        rows.append({
            "feature": column,
            "train_mean": round(train_mean, 6),
            "test_mean": round(test_mean, 6),
            "mean_shift_pct": round(float(mean_shift), 3),
            "train_std": round(train_std, 6),
            "test_std": round(test_std, 6),
            "std_shift_pct": round(float(std_shift), 3),
            "ks_stat": round(float(ks_stat), 6),
            "p_value": round(float(p_value), 6),
            "status": "Drift" if p_value < 0.05 else "No Drift",
            "drifted": 1 if p_value < 0.05 else 0,
        })

    drift_df = truerize.pl.DataFrame(rows).sort("p_value")
    drift_txt_path = BASE_DIR / "drift_reports" / "ks_drift_report.txt"
    drift_csv_path = BASE_DIR / "drift_reports" / "ks_drift_report.csv"
    with drift_txt_path.open("w", encoding="utf-8") as handle:
        for row in drift_df.to_dicts():
            handle.write(
                f"{row['feature']:<32} train_mean={row['train_mean']:<12} test_mean={row['test_mean']:<12} "
                f"mean_shift={row['mean_shift_pct']:<8} ks_stat={row['ks_stat']:<8} p_value={row['p_value']:<8} status={row['status']}\n"
            )
    drift_df.write_csv(drift_csv_path)

    _plot_cdf_grid(train_raw, test_raw, ROOM_TYPE_NUMERIC_FEATURES, BASE_DIR / "ks_reports" / "cdf_grid.png")
    _plot_kde_grid(train_raw, test_raw, ROOM_TYPE_NUMERIC_FEATURES, BASE_DIR / "ks_reports" / "kde_grid.png")
    return drift_df


def _plot_cdf_grid(train_df, test_df, columns, save_path):
    fig, axes = truerize.plt.subplots(2, 4, figsize=(18, 9))
    axes = axes.flatten()
    for index, column in enumerate(columns):
        train_sorted = truerize.np.sort(train_df[column].to_numpy())
        test_sorted = truerize.np.sort(test_df[column].to_numpy())
        train_cdf = truerize.np.arange(1, len(train_sorted) + 1) / len(train_sorted)
        test_cdf = truerize.np.arange(1, len(test_sorted) + 1) / len(test_sorted)
        axes[index].plot(train_sorted, train_cdf, label="Train")
        axes[index].plot(test_sorted, test_cdf, linestyle="--", label="Test")
        axes[index].set_title(column)
        axes[index].grid(True, alpha=0.3)
        axes[index].legend()
    fig.suptitle("Airbnb Feature CDF Comparison (Train vs Test)", fontsize=16)
    fig.tight_layout()
    fig.savefig(save_path, dpi=220, bbox_inches="tight")
    truerize.plt.close(fig)


def _plot_kde_grid(train_df, test_df, columns, save_path):
    fig, axes = truerize.plt.subplots(2, 4, figsize=(18, 9))
    axes = axes.flatten()
    for index, column in enumerate(columns):
        train_arr = train_df[column].to_numpy()
        test_arr = test_df[column].to_numpy()
        if len(train_arr) < 2 or len(test_arr) < 2:
            continue
        train_kde = truerize.gaussian_kde(train_arr)
        test_kde = truerize.gaussian_kde(test_arr)
        x_values = truerize.np.linspace(min(train_arr.min(), test_arr.min()), max(train_arr.max(), test_arr.max()), 500)
        axes[index].plot(x_values, train_kde(x_values), label="Train")
        axes[index].plot(x_values, test_kde(x_values), linestyle="--", label="Test")
        axes[index].set_title(column)
        axes[index].grid(True, alpha=0.3)
        axes[index].legend()
    fig.suptitle("Airbnb Feature KDE Comparison (Train vs Test)", fontsize=16)
    fig.tight_layout()
    fig.savefig(save_path, dpi=220, bbox_inches="tight")
    truerize.plt.close(fig)
# APP_EXPORT_END

drift_df = compute_drift_reports(room_type_split)
print(drift_df)

shape: (8, 11)
+--------------------------------+-------------+-------------+----------------+------------+------------+---------------+----------+----------+----------+---------+
| feature                        | train_mean  | test_mean   | mean_shift_pct | train_std  | test_std   | std_shift_pct | ks_stat  | p_value  | status   | drifted |
| ---                            | ---         | ---         | ---            | ---        | ---        | ---           | ---      | ---      | ---      | ---     |
| str                            | f64         | f64         | f64            | f64        | f64        | f64           | f64      | f64      | str      | i64     |
+==================================================================================================================================================================+
| availability_365               | 134.718623  | 132.805886  | -1.42          | 129.568705 | 129.425239 | -0.111        | 0.008928 | 0.161668 | No Drift | 0    

### Model Storage Paths

Defines the file paths for saving trained machine learning models, providing a centralized location for storing and loading model artifacts.

In [29]:
ROOM_TYPE_MODEL_PATHS = {
    "Logistic Regression": BASE_DIR / "outputs" / "models" / "room_type_logistic.pkl",
    "Random Forest": BASE_DIR / "outputs" / "models" / "room_type_rf.pkl",
    "XGBoost": BASE_DIR / "outputs" / "models" / "room_type_xgb.pkl",
    "LightGBM": BASE_DIR / "outputs" / "models" / "room_type_lgb.pkl",
    "SVM": BASE_DIR / "outputs" / "models" / "room_type_svc.pkl",
}

In [30]:
model_dir = BASE_DIR / 'outputs' / 'models'
model_dir.mkdir(parents=True, exist_ok=True)

feat_cols = room_type_split["feature_columns"]
x_train = room_type_split["X_train"]
x_test = room_type_split["X_test"]
y_train_enc = room_type_split["y_train"]
y_test_enc = room_type_split["y_test"]
label_encoder = room_type_split["label_encoder"]

MODEL_PATHS = ROOM_TYPE_MODEL_PATHS

print('Feature count   :', len(feat_cols))
print('Classes         :', list(label_encoder.classes_))

Feature count   : 22
Classes         : ['Entire home/apt', 'Hotel room', 'Private room', 'Shared room']


### Logistic Regression

The Logistic Regression model was trained on the standardized training dataset, evaluated on the test dataset using classification metrics, and saved for future predictions.

In [31]:
truerize.warnings.filterwarnings('ignore')

lr_model = truerize.LogisticRegression(class_weight='balanced', max_iter=1200)
lr_model.fit(x_train, y_train_enc)
preds = lr_model.predict(x_test)
print('===== Logistic Regression =====')
print('Accuracy:', truerize.accuracy_score(y_test_enc, preds))
print('Precision:', truerize.metrics.precision_score(y_test_enc, preds, average='weighted', zero_division=0))
print('\nClassification Report:')
print(truerize.classification_report(y_test_enc, preds, zero_division=0))
truerize.joblib.dump(lr_model, MODEL_PATHS['Logistic Regression'])

===== Logistic Regression =====
Accuracy: 0.3388228118328759
Precision: 0.5521937105679231

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.30      0.40     10277
           1       0.00      0.74      0.01        23
           2       0.51      0.39      0.44      8945
           3       0.03      0.31      0.06       429

    accuracy                           0.34     19674
   macro avg       0.29      0.43      0.23     19674
weighted avg       0.55      0.34      0.41     19674



['/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/models/room_type_logistic.pkl']

### Random Forest

The Random Forest model was trained on the standardized training dataset, evaluated on the test dataset using classification metrics, and saved for future predictions.

In [32]:
rf_model = truerize.RandomForestClassifier(n_estimators=240, max_depth=8, class_weight='balanced', random_state=SEED)
rf_model.fit(x_train, y_train_enc)
preds = rf_model.predict(x_test)
print('===== Random Forest =====')
print('Accuracy:', truerize.accuracy_score(y_test_enc, preds))
print('Precision:', truerize.metrics.precision_score(y_test_enc, preds, average='weighted', zero_division=0))
print('\nClassification Report:')
print(truerize.classification_report(y_test_enc, preds, zero_division=0))
truerize.joblib.dump(rf_model, MODEL_PATHS['Random Forest'])

===== Random Forest =====
Accuracy: 0.5556063840601809
Precision: 0.6479110171213279

Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.70      0.69     10277
           1       0.04      0.78      0.08        23
           2       0.64      0.39      0.48      8945
           3       0.08      0.61      0.14       429

    accuracy                           0.56     19674
   macro avg       0.36      0.62      0.35     19674
weighted avg       0.65      0.56      0.58     19674



['/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/models/room_type_rf.pkl']

### XGBoost

The XGBoost model was trained on the standardized training dataset using balanced sample weights, evaluated on the test dataset using classification metrics, and saved for future predictions.

In [33]:
class_weights = dict(
    zip(
        truerize.np.unique(y_train_enc),
        truerize.compute_class_weight(class_weight='balanced', classes=truerize.np.unique(y_train_enc), y=y_train_enc),
    )
)
sample_weights = truerize.np.array([class_weights[label] for label in y_train_enc])

xgb_model = truerize.XGBClassifier(n_estimators=220, max_depth=5, learning_rate=0.05, random_state=SEED, verbosity=0, eval_metric='mlogloss')
xgb_model.fit(x_train, y_train_enc, sample_weight=sample_weights)
preds = xgb_model.predict(x_test)
print('===== XGBoost =====')
print('Accuracy:', truerize.accuracy_score(y_test_enc, preds))
print('Precision:', truerize.metrics.precision_score(y_test_enc, preds, average='weighted', zero_division=0))
print('\nClassification Report:')
print(truerize.classification_report(y_test_enc, preds, zero_division=0))
truerize.joblib.dump(xgb_model, MODEL_PATHS['XGBoost'])

===== XGBoost =====
Accuracy: 0.5884415980481854
Precision: 0.6669376049625272

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.68      0.70     10277
           1       0.07      0.74      0.13        23
           2       0.65      0.48      0.55      8945
           3       0.09      0.66      0.17       429

    accuracy                           0.59     19674
   macro avg       0.38      0.64      0.39     19674
weighted avg       0.67      0.59      0.62     19674



['/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/models/room_type_xgb.pkl']

### LightGBM

The LightGBM model was trained on the standardized training dataset with balanced class weights, evaluated on the test dataset using classification metrics, and saved for future predictions.

In [34]:
lgb_model = truerize.LGBMClassifier(n_estimators=220, max_depth=5, learning_rate=0.05, class_weight='balanced', random_state=SEED, verbose=-1)
lgb_model.fit(x_train, y_train_enc)
preds = lgb_model.predict(x_test)
print('===== LightGBM =====')
print('Accuracy:', truerize.accuracy_score(y_test_enc, preds))
print('Precision:', truerize.metrics.precision_score(y_test_enc, preds, average='weighted', zero_division=0))
print('\nClassification Report:')
print(truerize.classification_report(y_test_enc, preds, zero_division=0))
truerize.joblib.dump(lgb_model, MODEL_PATHS['LightGBM'])

===== LightGBM =====
Accuracy: 0.6027752363525465
Precision: 0.6716753961599117

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.69      0.70     10277
           1       0.10      0.61      0.17        23
           2       0.65      0.50      0.57      8945
           3       0.11      0.67      0.18       429

    accuracy                           0.60     19674
   macro avg       0.39      0.62      0.40     19674
weighted avg       0.67      0.60      0.63     19674



['/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/models/room_type_lgb.pkl']

### Support Vector Machine (SVM)

The Support Vector Machine (SVM) model was trained on the standardized training dataset with balanced class weights, evaluated on the test dataset using classification metrics, and saved for future predictions.

In [35]:
svc_model = truerize.SVC(kernel='rbf',  class_weight='balanced', random_state=SEED)
svc_model.fit(x_train, y_train_enc)
preds = svc_model.predict(x_test)
print('===== SVM =====')
print('Accuracy:', truerize.accuracy_score(y_test_enc, preds))
print('Precision:', truerize.metrics.precision_score(y_test_enc, preds, average='weighted', zero_division=0))
print('\nClassification Report:')
print(truerize.classification_report(y_test_enc, preds, zero_division=0))
truerize.joblib.dump(svc_model, MODEL_PATHS['SVM'])

===== SVM =====
Accuracy: 0.4952729490698384
Precision: 0.5950122448818176

Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.52      0.58     10277
           1       0.02      0.26      0.04        23
           2       0.54      0.47      0.50      8945
           3       0.06      0.52      0.11       429

    accuracy                           0.50     19674
   macro avg       0.32      0.44      0.31     19674
weighted avg       0.60      0.50      0.54     19674



['/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/models/room_type_svc.pkl']

### Model Comparison

The performance of all trained models was compared using test accuracy. The models were ranked, and the best-performing model was identified based on the highest accuracy.

In [36]:
results = [
    {'model': 'Logistic Regression', 'accuracy': truerize.accuracy_score(y_test_enc, lr_model.predict(x_test))},
    {'model': 'Random Forest', 'accuracy': truerize.accuracy_score(y_test_enc, rf_model.predict(x_test))},
    {'model': 'XGBoost', 'accuracy': truerize.accuracy_score(y_test_enc, xgb_model.predict(x_test))},
    {'model': 'LightGBM', 'accuracy': truerize.accuracy_score(y_test_enc, lgb_model.predict(x_test))},
    {'model': 'SVM', 'accuracy': truerize.accuracy_score(y_test_enc, svc_model.predict(x_test))},
]
results_df = truerize.pl.DataFrame(results).sort('accuracy', descending=True)
print('===== MODEL COMPARISON =====')
print(results_df)
best_model = results_df.row(0)
print('\n===== BEST MODEL =====')
print(f'Model   : {best_model[0]}')
print(f'Accuracy: {round(best_model[1], 4)}')

===== MODEL COMPARISON =====
shape: (5, 2)
+---------------------+----------+
| model               | accuracy |
| ---                 | ---      |
| str                 | f64      |
+================================+
| LightGBM            | 0.602775 |
|---------------------+----------|
| XGBoost             | 0.588442 |
|---------------------+----------|
| Random Forest       | 0.555606 |
|---------------------+----------|
| SVM                 | 0.495273 |
|---------------------+----------|
| Logistic Regression | 0.338823 |
+---------------------+----------+

===== BEST MODEL =====
Model   : LightGBM
Accuracy: 0.6028


### SHAP Explainability Analysis

SHAP (SHapley Additive exPlanations) was used to interpret the trained LightGBM model by generating local feature contribution plots and global feature importance visualizations. The analysis produced force plots, beeswarm plots, and bar plots for each room type class.

In [37]:
# APP_EXPORT_START
def run_full_shap_analysis(model, split, row_index=0, sample_size=220, output_dir=BASE_DIR / "outputs" / "shap"):
    output_dir.mkdir(parents=True, exist_ok=True)
    x_test = split["X_test"]
    feature_names = split["feature_columns"]
    class_names = split["label_encoder"].classes_

    explainer = truerize.shap.Explainer(model)
    shap_values = explainer(x_test)

    saved_paths = []
    for class_index, class_name in enumerate(class_names):
        row_shap = shap_values.values[row_index][:, class_index]
        base_value = shap_values.base_values[row_index][class_index]
        truerize.plt.figure(figsize=(18, 5.8))
        truerize.shap.force_plot(
            base_value, row_shap, x_test.iloc[row_index],
            feature_names=feature_names, matplotlib=True, show=False
        )
        local_path = output_dir / f"local_class_{class_name.replace('/', '_')}.png"
        truerize.plt.tight_layout()
        truerize.plt.savefig(local_path, dpi=260, bbox_inches="tight")
        saved_paths.append(local_path)
        truerize.plt.close()

    sample = x_test.iloc[: min(sample_size, len(x_test))]
    sample_shap = explainer(sample)
    for class_index, class_name in enumerate(class_names):
        values = sample_shap.values[:, :, class_index]
        safe_name = class_name.replace('/', '_')

        truerize.plt.figure(figsize=(16, 10))
        truerize.shap.summary_plot(values, sample, feature_names=feature_names, show=False, plot_size=(16, 10))
        beeswarm_path = output_dir / f"global_beeswarm_{safe_name}.png"
        truerize.plt.savefig(beeswarm_path, dpi=260, bbox_inches="tight")
        saved_paths.append(beeswarm_path)
        truerize.plt.close()

        truerize.plt.figure(figsize=(14, 9))
        truerize.shap.summary_plot(values, sample, feature_names=feature_names, plot_type="bar", show=False, plot_size=(14, 9))
        bar_path = output_dir / f"global_bar_{safe_name}.png"
        truerize.plt.savefig(bar_path, dpi=260, bbox_inches="tight")
        saved_paths.append(bar_path)
        truerize.plt.close()

    return saved_paths, shap_values
# APP_EXPORT_END

shap_paths, shap_values = run_full_shap_analysis(lgb_model, room_type_split)
print([str(path) for path in shap_paths])

['/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/shap/local_class_Entire home_apt.png', '/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/shap/local_class_Hotel room.png', '/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/shap/local_class_Private room.png', '/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/shap/local_class_Shared room.png', '/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/shap/global_beeswarm_Entire home_apt.png', '/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/shap/global_bar_Entire home_apt.png', '/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/shap/global_beeswarm_Hotel room.png', '/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/shap/global_bar_Hotel room.png', '/Users/vishwajeetsiranje/Deskt

<Figure size 1800x580 with 0 Axes>

<Figure size 1800x580 with 0 Axes>

<Figure size 1800x580 with 0 Axes>

<Figure size 1800x580 with 0 Axes>

### LIME Explainability Analysis

LIME (Local Interpretable Model-agnostic Explanations) was used to explain individual predictions made by the trained LightGBM model. Local feature importance plots were generated for each room type class to visualize the contribution of the most influential features.

In [38]:
# APP_EXPORT_START
def run_lime_analysis(model, split, sample_index=0, output_dir=BASE_DIR / "outputs" / "lime"):
    output_dir.mkdir(parents=True, exist_ok=True)
    feature_names = split["feature_columns"]
    class_names = split["label_encoder"].classes_

    explainer = truerize.LimeTabularExplainer(
        training_data=split["X_train"].to_numpy(),
        feature_names=feature_names,
        class_names=class_names.tolist(),
        mode="classification",
        discretize_continuous=True,
        random_state=SEED,
    )

    sample = split["X_test"].iloc[sample_index].to_numpy()

    explanation = explainer.explain_instance(
        data_row=sample,
        predict_fn=model.predict_proba,
        num_features=min(len(feature_names), 12),
        labels=list(range(len(class_names))),
    )

    saved_paths = []
    for class_index, class_name in enumerate(class_names):
        fig = explanation.as_pyplot_figure(label=class_index)
        fig.set_size_inches(13.5, 7.5)
        safe_name = class_name.replace('/', '_')
        path = output_dir / f"lime_class_{safe_name}.png"
        fig.savefig(path, dpi=260, bbox_inches="tight")
        saved_paths.append(path)
        truerize.plt.close(fig)
    return saved_paths
# APP_EXPORT_END

lime_paths = run_lime_analysis(lgb_model, room_type_split)
print([str(path) for path in lime_paths])

['/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/lime/lime_class_Entire home_apt.png', '/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/lime/lime_class_Hotel room.png', '/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/lime/lime_class_Private room.png', '/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/lime/lime_class_Shared room.png']


### XAI Report Generation

A comprehensive Explainable AI (XAI) report was generated in Microsoft Word format. The report includes the input feature values, predicted room type, class probabilities, SHAP-based feature contributions, natural-language explanations, visual explanations (SHAP and LIME), and a summary of the model's prediction.

In [ ]:
# APP_EXPORT_START
def generate_xai_report(model, split, shap_values=None, sample_index=0, output_path=BASE_DIR / "outputs" / "xai_report" / "xai_report.docx"):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    feature_names = split["feature_columns"]
    label_encoder = split["label_encoder"]
    x_test = split["X_test"]

    doc = truerize.Document()
    doc.styles["Normal"].paragraph_format.space_after = truerize.Inches(0.03)
    doc.add_heading("Airbnb Room Type Classification - XAI Report", 0)

    sample = x_test.iloc[sample_index]
    doc.add_heading("1. Input Feature Snapshot", level=1)
    for name, value in zip(feature_names, sample):
        doc.add_paragraph(f"{name}: {round(float(value), 4)}")

    sample_2d = sample.to_numpy().reshape(1, -1)
    pred_encoded = int(model.predict(sample_2d)[0])
    probabilities = model.predict_proba(sample_2d)[0]
    predicted_label = label_encoder.inverse_transform([pred_encoded])[0]
    confidence = float(max(probabilities))

    doc.add_heading("2. Prediction", level=1)
    doc.add_paragraph(f"Predicted room type: {predicted_label}")
    doc.add_paragraph(f"Confidence: {round(confidence, 4)}")
    doc.add_paragraph("Class probabilities:")
    for class_name, prob in zip(label_encoder.classes_, probabilities):
        doc.add_paragraph(f"  {class_name}: {round(float(prob), 4)}")

    if shap_values is None:
        explainer = truerize.shap.Explainer(model)
        shap_values = explainer(x_test)

    class_index = pred_encoded
    row_shap = shap_values.values[sample_index][:, class_index]
    doc.add_heading("3. Top SHAP Drivers For Predicted Class", level=1)
    top_items = sorted(zip(feature_names, sample.to_list(), row_shap), key=lambda item: abs(item[2]), reverse=True)[:5]
    for feature_name, value, contribution in top_items:
        direction = f"pushes toward {predicted_label}" if contribution > 0 else f"pushes away from {predicted_label}"
        doc.add_paragraph(f"{feature_name} = {round(float(value), 4)} -> {direction}")

    doc.add_heading("4. Natural-Language Explanation", level=1)
    for feature_name, value, contribution in top_items[:3]:
        direction = f"supporting the {predicted_label} prediction" if contribution > 0 else f"working against the {predicted_label} prediction"
        doc.add_paragraph(f"{feature_name} is {direction}, with observed value {round(float(value), 4)}.")

    def add_image_block(title, directory, token):
        if not directory.exists():
            return
        for image_path in sorted(directory.iterdir()):
            if token in image_path.name and image_path.suffix.lower() == ".png":
                doc.add_paragraph(title)
                doc.add_picture(str(image_path), width=truerize.Inches(4.8))

    doc.add_heading("5. Visual Explanations", level=1)
    add_image_block("SHAP Local / Global", BASE_DIR / "outputs" / "shap", "")
    add_image_block("LIME Views", BASE_DIR / "outputs" / "lime", "lime")

    doc.add_heading("6. Final Insight", level=1)
    doc.add_paragraph(
        f"This listing is classified as {predicted_label} with {round(confidence * 100, 1)}% confidence. "
        f"Given the overall model accuracy of ~60% and the weak feature-target correlations identified during "
        f"data validation, this explanation reflects the strongest available signal in the dataset rather than "
        f"a highly confident, well-separated prediction."
    )

    doc.save(output_path)
    return output_path
# APP_EXPORT_END

xai_report_path = generate_xai_report(lgb_model, room_type_split, shap_values=shap_values)
print(xai_report_path)

/Users/vishwajeetsiranje/Desktop/AIR_BNB_PREDICTION/Airbnb_Data_Prediction/outputs/xai_report/xai_report.docx
